In [1]:
# import os
# os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
# os.environ["CUDA_VISIBLE_DEVICES"]="1"

import datasets
import json
import numpy as np
import pandas as pd
from collections import defaultdict

/home/artem.shelmanov/miniconda3/envs/dllm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# from lm_polygraph.generation_metrics.alignscore import AlignScore
from lm_polygraph.ue_metrics.ue_metric import get_random_scores
# from tqdm import tqdm

# scorer_sum = AlignScore(target_is_claims=False)

/home/artem.shelmanov/miniconda3/envs/dllm/lib/python3.11/site-packages/torch/cuda/__init__.py:1061: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()


In [3]:
from lm_polygraph.ue_metrics.pred_rej_area import PredictionRejectionArea

prr = PredictionRejectionArea(0.5)

In [4]:
# import torch
# import torch.nn.functional as F

# from transformers import AutoTokenizer, AutoModel


# def forward_process(batch, prompt_index, mask_id):
#     b, l = batch.shape

#     target_len = (l - prompt_index.sum()).item()
#     k = torch.randint(1, target_len + 1, (), device=batch.device)

#     x = torch.round(torch.linspace(float(k), k + (b - 1) * (target_len / b), steps=b, device=batch.device)).long()
#     x = ((x - 1) % target_len) + 1
#     assert x.min() >= 1 and x.max() <= target_len

#     indices = torch.arange(target_len, device=batch.device).repeat(b, 1)
#     is_mask = indices < x.unsqueeze(1)
#     for i in range(b):
#         is_mask[i] = is_mask[i][torch.randperm(target_len)]

#     is_mask = torch.cat((torch.zeros(b, prompt_index.sum(), dtype=torch.bool, device=batch.device), is_mask), dim=1)
#     noisy_batch = torch.where(is_mask, mask_id, batch)

#     # Return the masked batch and the mask ratio
#     return noisy_batch, (x / target_len).unsqueeze(1).repeat(1, l)


# def get_logits(model, batch, prompt_index, cfg_scale, mask_id):
#     if cfg_scale > 0.:
#         assert len(prompt_index) == batch.shape[1]
#         prompt_index = prompt_index.unsqueeze(0).repeat(batch.shape[0], 1)
#         un_batch = batch.clone()
#         un_batch[prompt_index] = mask_id
#         batch = torch.cat([batch, un_batch])

#     input = batch
#     logits = model(input).logits

#     if cfg_scale > 0.:
#         logits, un_logits = torch.chunk(logits, 2, dim=0)
#         logits = un_logits + (cfg_scale + 1) * (logits - un_logits)
#     return logits


# @ torch.no_grad()
# def get_log_likelihood(model, prompt, answer, mc_num=128, batch_size=16, cfg_scale=0., mask_id=126336):
#     '''
#     Args:
#         model: Mask predictor.
#         prompt: A tensor of shape (l1).
#         answer: A tensor of shape (l2).
#         mc_num: Monte Carlo estimation times.
#                 As detailed in Appendix B.5. Since MMLU, CMMLU, and C-EVAL only require the likelihood of a single token, a
#                 single Monte Carlo estimate is sufficient for these benchmarks. For all other benchmarks, we find that 128
#                 Monte Carlo samples are adequate to produce stable results.
#         batch_size: Mini batch size.
#         cfg_scale: Unsupervised classifier-free guidance scale.
#         mask_id: The toke id of [MASK] is 126336.
#     '''
#     seq = torch.concatenate([prompt, answer])[None, :]
#     seq = seq.repeat((batch_size, 1)).to(model.device)
#     prompt_index = torch.arange(seq.shape[1], device=model.device) < len(prompt)

#     loss_ = []
#     for _ in range(mc_num // batch_size):
#         perturbed_seq, p_mask = forward_process(seq, prompt_index, mask_id)
#         mask_index = perturbed_seq == mask_id

#         logits = get_logits(model, perturbed_seq, prompt_index, cfg_scale, mask_id)

#         loss = F.cross_entropy(logits[mask_index], seq[mask_index], reduction='none') / p_mask[mask_index]
#         loss = loss.sum() / batch_size

#         loss_.append(loss.item())

#     return - sum(loss_) / len(loss_)

In [5]:
# device = 'cuda'

# llada_model = AutoModel.from_pretrained('GSAI-ML/LLaDA-1.5', trust_remote_code=True, torch_dtype=torch.bfloat16, device_map="auto").eval()
# tokenizer = AutoTokenizer.from_pretrained('GSAI-ML/LLaDA-1.5', trust_remote_code=True)

In [6]:
import re
import torch
from sklearn.metrics import roc_auc_score
import string

def preproc_method_name(x):
    return x

def get_table_short(path, metric_names=["Rouge_rougeL", 'AlignScore'], dataset="CoQA", ue_metric="prr_0.5_normalized", update_alignscore_sum=False,
              update_alignscore=False, aggregated=False, return_score=False, level="sequence", upd=False, roc_auc=False, upd_med=False, add_nll=False, mc_num=16):
    
    man = torch.load(path, weights_only=False)
    df = {}
    
    methods = np.unique([x[1] for x in man['metrics'].keys() if "2side" not in x[1]])
    if add_nll:
        methods = np.array(list(methods) + ["SampleNLL"])
        
        greedy_text = np.array(man["stats"][("greedy_texts")])
        input_text = np.array(man["stats"][("input_texts")])

        ue = []
        for i in tqdm(range(len(greedy_text))):
            prompt = torch.tensor(tokenizer(input_text[i])['input_ids']).to(llada_model.device)
            answer = torch.tensor(tokenizer(greedy_text[i])['input_ids']).to(llada_model.device)
            if len(answer):
                ue.append(-get_log_likelihood(llada_model, prompt, answer, mc_num=mc_num))
            else: 
                ue.append(1e+3)
        sample_nll = np.array(ue)
    
    if update_alignscore_sum and (dataset in ['XSUM', 'SamSum', 'CNN']):
        greedy_texts = [x if len(x.strip())>0 else "-" for x in man['stats']["greedy_texts"]]
        greedy_texts = np.array(greedy_texts)
        target_texts = man['stats']["target_texts"]
        input_texts = man['stats']["input_texts"]
        if "Summary (one sentence):" in input_texts[0]:
            context = np.array([x.split("Text:")[-1].split("Summary (one sentence):")[0] for x in input_texts])
        elif "End of dialogue" in input_texts[0]:
            context = np.array([x.split("Here\'s the dialogue:\n\n")[-1].split(" (End of dialogue)")[0] for x in input_texts])
        elif "End of text" in input_texts[0]:
            context = np.array([x.split("Here\'s the text:\n\n")[-1].split(" (End of text)")[0] for x in input_texts])
        alignscores_0 = scorer_sum({"greedy_texts": greedy_texts, "target_texts": context}, context)

    if (dataset in ['GSM8k']):
        greedy_texts = [x if len(x.strip())>0 else "-" for x in man['stats']["greedy_texts"]]
        greedy_texts = np.array(greedy_texts)
        target_texts = np.array(man['stats']["target_texts"])
        acc = np.array(man["gen_metrics"][("sequence", "Accuracy")])

        answers = []
        for text in greedy_texts[acc == 0]:
            answer_match = re.search(
                r'(?:The answer is|Answer[:]?)[^\d]*(\d+)',
                text, 
                flags=re.IGNORECASE
            )
            try:
                answer = answer_match.group(1).strip()
                answer = answer.translate(str.maketrans("", "", string.punctuation))
                answers.append(int(answer))
            except:
                answers.append(-1e10) 
        answers = np.array(answers)
        targets = []
        for text in target_texts[acc == 0]:
            output_ignore_regex = re.compile("(?s).*#### ")
            answer = output_ignore_regex.sub("", text)
            answer = answer.translate(str.maketrans("", "", string.punctuation))
            targets.append(int(answer)) 
        targets = np.array(targets)

        acc[acc == 0] = (answers == targets) * 1

        
    for method in methods:
        for metric in metric_names:
            key = (level, method, metric, ue_metric)
            scores = df.get(method, [])
            # if method == "NFEUncertainty":
            #     ue = -np.array([x if not isinstance(x, complex) else x.real for x in man['estimations'][(level, method)]])
            #     metrics = np.array(man['gen_metrics'][(level, metric)])
            #     mean_val = prr(ue, metrics) 
            #     oracle = prr(-metrics, metrics)
            #     random = get_random_scores(prr, metrics)
            #     scores.append((mean_val - random) / (oracle - random))
            if method == "SampleNLL":
                ue = sample_nll
                if update_alignscore_sum and (dataset in ['XSUM', 'SamSum', 'CNN']):
                    metrics = alignscores_0
                elif (dataset in ['GSM8k']) and (metric == "Accuracy"):
                    metrics = acc
                else:
                    metrics = np.array(man['gen_metrics'][(level, metric)])
                mean_val = prr(ue, metrics) 
                oracle = prr(-metrics, metrics)
                random = get_random_scores(prr, metrics)
                scores.append((mean_val - random) / (oracle - random))

            elif update_alignscore_sum and (dataset in ['XSUM', 'SamSum', 'CNN']):
                metrics = np.array(man['gen_metrics'][(level, metric)])
                ue = np.array([x if not isinstance(x, complex) else x.real for x in man['estimations'][(level, method)]])
                metrics = alignscores_0
                mean_val = prr(ue, metrics) 
                oracle = prr(-metrics, metrics)
                random = get_random_scores(prr, metrics)
                scores.append((mean_val - random) / (oracle - random))
            elif (dataset in ['GSM8k']) and (metric == "Accuracy"):
                metrics = acc
                ue = np.array([x if not isinstance(x, complex) else x.real for x in man['estimations'][(level, method)]])
                mean_val = prr(ue, metrics) 
                oracle = prr(-metrics, metrics)
                random = get_random_scores(prr, metrics)
                scores.append((mean_val - random) / (oracle - random))
            else:
                scores.append(man['metrics'][key])
            df[method] = scores
    for metric in metric_names:
        if update_alignscore_sum and (dataset in ['XSUM', 'SamSum', 'CNN']):
            print(f"{metric}: ", np.mean(alignscores_0))
        elif (dataset in ['GSM8k']) and (metric == "Accuracy"):
            print(f"{metric}: ", np.mean(acc))
        else:
            print(f"{metric}: ", np.mean(man['gen_metrics'][("sequence", metric)]))
    df0 = pd.DataFrame(df, index=pd.MultiIndex.from_tuples([(dataset, m) for m in metric_names])).T
    df0.loc[:, (dataset, 'Method')] = [x for x in df0.index]
    return df0.reset_index(drop=True)

In [6]:
import os
import copy
from scipy.stats import rankdata

base_path = "./workdir/output"
base_path = "/home/jovyan/artemvazhentsev/llm-tad-uncertainty/workdir/output_unsup_may25_final"

datasets = ['ats/xsum', 'ats/samsum', "ats/['cnn_dailymail', '3.0.0']", "nmt/['wmt14', 'fr-en']", "nmt/['wmt19', 'de-en']", "qa/['truthful_qa', 'generation']", 'qa/coqa', 'qa/sciq', "qa/['trivia_qa', 'rc.nocontext']", "qa/['cais/mmlu', 'all']", "qa/['gsm8k', 'main']"]
datasets_names = ['XSUM', 'SamSum', 'CNN', "WMT14", "WMT19", "TruthfulQA", 'CoQA', 'SciQ', "TriviaQA", "MMLU", "GSM8k"]

# datasets = ['ats/samsum', "nmt/['wmt14', 'fr-en']", "nmt/['wmt19', 'de-en']", "qa/['trivia_qa', 'rc.nocontext']", "qa/['gsm8k', 'main']"]
# datasets_names = ['SamSum', "WMT14", "WMT19", "TriviaQA", "GSM8k"]

models = ['GSAI-ML/LLaDA-1.5', 'meta-llama/Llama-3.1-8B', 'meta-llama/Llama-3.1-8B-Instruct']

paths = {
    "GSAI-ML/LLaDA-1.5": {
        # "GSM8k": "./workdir/output_dllm/qa/GSAI-ML/LLaDA-1.5/['gsm8k', 'main']/2025-06-26/15-22-42/ue_manager_seed1",
        "GSM8k": "./workdir/output_dllm_dec25/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/gsm8k', 'simple_instruct']/2025-12-20/17-07-01/ue_manager_seed1",
        "CNN": "./workdir/output_dllm/ats/GSAI-ML/LLaDA-1.5/['cnn_dailymail', '3.0.0']/2025-06-28/08-10-10/ue_manager_seed1",
        "XSUM": "./workdir/output_dllm/ats/GSAI-ML/LLaDA-1.5/xsum/2025-06-29/14-26-07/ue_manager_seed1",
        # "SamSum": "./workdir/output_dllm/ats/GSAI-ML/LLaDA-1.5/samsum/2025-06-29/14-26-07/ue_manager_seed1",
        "SamSum": "./workdir/output_dllm_dec25/ats/GSAI-ML/LLaDA-1.5/['LM-Polygraph/samsum', 'simple_instruct']/2025-12-18/17-01-33/ue_manager_seed1",
        "CoQA": "./workdir/output_dllm/qa/GSAI-ML/LLaDA-1.5/coqa/2025-06-29/09-42-26/ue_manager_seed1",
        "SciQ": "./workdir/output_dllm/qa/GSAI-ML/LLaDA-1.5/sciq/2025-06-30/07-41-34/ue_manager_seed1",
        "TruthfulQA": "./workdir/output_dllm/qa/GSAI-ML/LLaDA-1.5/['truthful_qa', 'generation']/2025-06-30/07-41-34/ue_manager_seed1",
        # "WMT14": "./workdir/output_dllm/nmt/GSAI-ML/LLaDA-1.5/['wmt14', 'fr-en']/2025-06-30/11-50-18/ue_manager_seed1",
        "WMT14": "./workdir/output_dllm_dec25/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2025-12-19/16-57-43/ue_manager_seed1",
        # "WMT19": "./workdir/output_dllm/nmt/GSAI-ML/LLaDA-1.5/['wmt19', 'de-en']/2025-06-30/11-50-18/ue_manager_seed1",
        "WMT19": "./workdir/output_dllm_dec25/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt19', 'deen_simple_instruct']/2025-12-23/18-17-25/ue_manager_seed1",
        # "TriviaQA": "./workdir/output_dllm/qa/GSAI-ML/LLaDA-1.5/['trivia_qa', 'rc.nocontext']/2025-06-30/20-09-50/ue_manager_seed1",
        "TriviaQA": "./workdir/output_dllm_dec25/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/triviaqa', 'simple_instruct']/2025-12-17/16-31-56/ue_manager_seed1",
        "MMLU": "./workdir/output_dllm/qa/GSAI-ML/LLaDA-1.5/['cais/mmlu', 'all']/2025-06-30/20-09-50/ue_manager_seed1",
    },
    "meta-llama/Llama-3.1-8B-Instruct": {
        "WMT19": "/home/jovyan/artemvazhentsev/llm-tad-uncertainty/workdir/output_unsup_may25_final_instruct_/nmt/meta-llama/Llama-3.1-8B-Instruct/['wmt19', 'de-en']/2025-06-11/19-38-21/ue_manager_seed1",
        "GSM8k": "/home/jovyan/artemvazhentsev/llm-tad-uncertainty/workdir/output_unsup_may25_final_instruct_/qa/meta-llama/Llama-3.1-8B-Instruct/['gsm8k', 'main']/2025-06-11/12-23-57/ue_manager_seed1",
        "CoQA": "/home/jovyan/artemvazhentsev/llm-tad-uncertainty/workdir/output_unsup_may25_final_instruct_/qa/meta-llama/Llama-3.1-8B-Instruct/coqa/2025-06-01/11-51-41/ue_manager_seed1",
        "TriviaQA": "/home/jovyan/artemvazhentsev/llm-tad-uncertainty/workdir/output_unsup_may25_final_instruct_/qa/meta-llama/Llama-3.1-8B-Instruct/['trivia_qa', 'rc.nocontext']/2025-06-02/10-08-42/ue_manager_seed1",
        "SamSum": "/home/jovyan/artemvazhentsev/llm-tad-uncertainty/workdir/output_unsup_may25_final_instruct_/ats/meta-llama/Llama-3.1-8B-Instruct/samsum/2025-06-01/16-59-58/ue_manager_seed1",
        "WMT14": "/home/jovyan/artemvazhentsev/llm-tad-uncertainty/workdir/output_unsup_may25_final_instruct_/nmt/meta-llama/Llama-3.1-8B-Instruct/['wmt14', 'fr-en']/2025-06-01/09-39-16/ue_manager_seed1",
        "SciQ": "/home/jovyan/artemvazhentsev/llm-tad-uncertainty/workdir/output_unsup_may25_final_instruct_/qa/meta-llama/Llama-3.1-8B-Instruct/sciq/2025-05-31/17-55-02/ue_manager_seed1",
        "TruthfulQA": "/home/jovyan/artemvazhentsev/llm-tad-uncertainty/workdir/output_unsup_may25_final_instruct_/qa/meta-llama/Llama-3.1-8B-Instruct/['truthful_qa', 'generation']/2025-06-18/14-52-07/ue_manager_seed1",
        "MMLU": "/home/jovyan/artemvazhentsev/llm-tad-uncertainty/workdir/output_unsup_may25_final_instruct_/qa/meta-llama/Llama-3.1-8B-Instruct/['cais/mmlu', 'all']/2025-06-01/09-38-45/ue_manager_seed1",
        "MedQUAD": "/home/jovyan/artemvazhentsev/llm-tad-uncertainty/workdir/output_unsup_may25_final_instruct_/qa/meta-llama/Llama-3.1-8B-Instruct/keivalya/MedQuad-MedicalQnADataset/2025-06-18/14-11-14/ue_manager_seed1",
        "XSUM": "/home/jovyan/artemvazhentsev/llm-tad-uncertainty/workdir/output_unsup_may25_final_instruct_/ats/meta-llama/Llama-3.1-8B-Instruct/xsum/2025-06-20/18-37-07/ue_manager_seed1",
        "CNN": "/home/jovyan/artemvazhentsev/llm-tad-uncertainty/workdir/output_unsup_may25_final_instruct_/ats/meta-llama/Llama-3.1-8B-Instruct/['cnn_dailymail', '3.0.0']/2025-06-19/22-18-02/ue_manager_seed1",
    }
}

filename = "ue_manager_seed1"
dfs = {}

all_paths = []
for model in models:
    dfs[model] = {}
    for ds, ds_name in zip(datasets, datasets_names):
        task = ds.split("/")[0]
        ds = '/'.join(ds.split("/")[1:])
        path = f"{base_path}/{task}/{model}/{ds}/"
        final_path = None
        if model in paths.keys() and ds_name in paths[model].keys():
            final_path = paths[model][ds_name]
        else:
            for address, dirs, files in os.walk(path):
                for name in files:
                    full_path = os.path.join(address, name)
                    if full_path.endswith(filename):
                        final_path = copy.deepcopy(full_path)
        if final_path is not None:
            print(final_path)
            all_paths.append(final_path)
            if task == "ats":
                metric_names=['AlignScore']
            elif (task == "qa") and (ds_name in ["MMLU", "GSM8k"]):
                metric_names=['Accuracy']
            elif task == "qa":
                metric_names=["AlignScore"]
            elif task == "nmt":
                metric_names=['Comet']
            man = torch.load(final_path, weights_only=False)

            # if ds_name in ['XSUM', 'SamSum', 'CNN']:
            #     greedy_texts = [x if len(x.strip())>0 else "-" for x in man['stats']["greedy_texts"]]
            #     greedy_texts = np.array(greedy_texts)
            #     target_texts = man['stats']["target_texts"]
            #     input_texts = man['stats']["input_texts"]
            #     if "Summary (one sentence):" in input_texts[0]:
            #         context = np.array([x.split("Text:")[-1].split("Summary (one sentence):")[0] for x in input_texts])
            #     elif "End of dialogue" in input_texts[0]:
            #         context = np.array([x.split("Here\'s the dialogue:\n\n")[-1].split(" (End of dialogue)")[0] for x in input_texts])
            #     elif "End of text" in input_texts[0]:
            #         context = np.array([x.split("Here\'s the text:\n\n")[-1].split(" (End of text)")[0] for x in input_texts])
            #     score = scorer_sum({"greedy_texts": greedy_texts, "target_texts": context}, context)
            # else:
            score = man["gen_metrics"][("sequence", metric_names[0])]
            dfs[model][ds_name] = [np.mean(score)]

./workdir/output_dllm/ats/GSAI-ML/LLaDA-1.5/xsum/2025-06-29/14-26-07/ue_manager_seed1
./workdir/output_dllm_dec25/ats/GSAI-ML/LLaDA-1.5/['LM-Polygraph/samsum', 'simple_instruct']/2025-12-18/17-01-33/ue_manager_seed1
./workdir/output_dllm/ats/GSAI-ML/LLaDA-1.5/['cnn_dailymail', '3.0.0']/2025-06-28/08-10-10/ue_manager_seed1
./workdir/output_dllm_dec25/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2025-12-19/16-57-43/ue_manager_seed1
./workdir/output_dllm_dec25/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt19', 'deen_simple_instruct']/2025-12-23/18-17-25/ue_manager_seed1
./workdir/output_dllm/qa/GSAI-ML/LLaDA-1.5/['truthful_qa', 'generation']/2025-06-30/07-41-34/ue_manager_seed1
./workdir/output_dllm/qa/GSAI-ML/LLaDA-1.5/coqa/2025-06-29/09-42-26/ue_manager_seed1
./workdir/output_dllm/qa/GSAI-ML/LLaDA-1.5/sciq/2025-06-30/07-41-34/ue_manager_seed1
./workdir/output_dllm_dec25/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/triviaqa', 'simple_instruct']/2025-12-17/16-31-56/ue_manager_see

In [14]:
qual = pd.concat([pd.DataFrame(dfs[model], index=[model]) for model in models])
qual.loc[:, "Mean"] = qual[qual.columns].mean(axis=1)
qual

,XSUM,SamSum,CNN,WMT14,WMT19,TruthfulQA,CoQA,SciQ,TriviaQA,MMLU,GSM8k,Mean
GSAI-ML/LLaDA-1.5,0.004745,0.896339,0.412046,0.859268,0.848987,0.433265,0.802830,0.828366,0.459859,0.6540,0.700531,0.627294
meta-llama/Llama-3.1-8B,0.037730,0.193020,0.376015,0.862830,0.869864,0.320344,0.757700,0.848560,0.685978,0.5570,0.551933,0.550998
meta-llama/Llama-3.1-8B-Instruct,0.003875,0.126779,0.303488,0.868607,0.868037,0.529666,0.778873,0.867273,0.726541,0.6155,0.843063,0.593791


In [ ]:
import os
import copy
from scipy.stats import rankdata

base_path = "./workdir/output"

datasets = ['ats/xsum', 'ats/samsum', "ats/['cnn_dailymail', '3.0.0']", "nmt/['wmt14', 'fr-en']", "nmt/['wmt19', 'de-en']", "qa/keivalya/MedQuad-MedicalQnADataset", "qa/['truthful_qa', 'generation']", 'qa/coqa', 'qa/sciq', "qa/['trivia_qa', 'rc.nocontext']", "qa/['cais/mmlu', 'all']", "qa/['gsm8k', 'main']"]
datasets_names = ['XSUM', 'SamSum', 'CNN', "WMT14", "WMT19", "MedQUAD", "TruthfulQA", 'CoQA', 'SciQ', "TriviaQA", "MMLU", "GSM8k"]

# datasets = ['ats/xsum', 'ats/samsum', "ats/['cnn_dailymail', '3.0.0']", "nmt/['wmt19', 'de-en']", 'qa/coqa', 'qa/sciq', "qa/['gsm8k', 'main']", "qa/['truthful_qa', 'generation']", "qa/['cais/mmlu', 'all']"]
# datasets_names = ['XSUM', 'SamSum', 'CNN', "WMT19", 'CoQA', "SciQ", "GSM8k", "TruthfulQA", "MMLU"]

# datasets = ["nmt/['wmt14', 'fr-en']"]
# datasets_names = ["WMT14"]

# datasets = ["qa/['trivia_qa', 'rc.nocontext']"]
# datasets_names = ["TriviaQA"]

datasets = ['ats/samsum', "nmt/['wmt14', 'fr-en']", "nmt/['wmt19', 'de-en']", "qa/['trivia_qa', 'rc.nocontext']", "qa/['gsm8k', 'main']"]
datasets_names = ['SamSum', "WMT14", "WMT19", "TriviaQA", "GSM8k"]

models = ['GSAI-ML/LLaDA-1.5/']

paths = {
    "GSAI-ML/LLaDA-1.5/": {
        # "WMT19": "./workdir/output/nmt/GSAI-ML/LLaDA-1.5/['wmt19', 'de-en']/2025-06-22/11-33-11/ue_manager_seed1",
        # "CoQA": "./workdir/output/qa/GSAI-ML/LLaDA-1.5/['LM-polygraph/coqa', 'continuation']/2025-06-20/20-54-52/ue_manager_seed1",
        # "GSM8k": "./workdir/output/qa/GSAI-ML/LLaDA-1.5/['gsm8k', 'main']/2025-06-22/13-05-43/ue_manager_seed1",
        # "TruthfulQA": "./workdir/output/qa/GSAI-ML/LLaDA-1.5/['truthful_qa', 'generation']/2025-06-23/10-06-41/ue_manager_seed1",
        # "MMLU": "./workdir/output/qa/GSAI-ML/LLaDA-1.5/['cais/mmlu', 'all']/2025-06-23/16-05-45/ue_manager_seed1",
        # "SamSum": "./workdir/output/ats/GSAI-ML/LLaDA-1.5/samsum/2025-06-23/20-37-07/ue_manager_seed1",
        # "XSUM": "./workdir/output/ats/GSAI-ML/LLaDA-1.5/xsum/2025-06-23/19-07-50/ue_manager_seed1",
        ###########################################
        # "GSM8k": "./workdir/output_dllm/qa/GSAI-ML/LLaDA-1.5/['gsm8k', 'main']/2025-06-26/15-22-42/ue_manager_seed1",
        # "GSM8k": "./workdir/output/qa/GSAI-ML/LLaDA-8B-Instruct/['gsm8k', 'main']/2025-12-11/21-13-27/ue_manager_seed1", 
        "GSM8k": "./workdir/output_dllm_dec25/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/gsm8k', 'simple_instruct']/2025-12-20/17-07-01/ue_manager_seed1",

        "CNN": "./workdir/output_dllm/ats/GSAI-ML/LLaDA-1.5/['cnn_dailymail', '3.0.0']/2025-06-28/08-10-10/ue_manager_seed1",
        "XSUM": "./workdir/output_dllm/ats/GSAI-ML/LLaDA-1.5/xsum/2025-06-29/14-26-07/ue_manager_seed1",
        # "SamSum": "./workdir/output_dllm/ats/GSAI-ML/LLaDA-1.5/samsum/2025-06-29/14-26-07/ue_manager_seed1",
        "SamSum": "./workdir/output_dllm_dec25/ats/GSAI-ML/LLaDA-1.5/['LM-Polygraph/samsum', 'simple_instruct']/2025-12-18/17-01-33/ue_manager_seed1",
        "CoQA": "./workdir/output_dllm/qa/GSAI-ML/LLaDA-1.5/coqa/2025-06-29/09-42-26/ue_manager_seed1",
        "SciQ": "./workdir/output_dllm/qa/GSAI-ML/LLaDA-1.5/sciq/2025-06-30/07-41-34/ue_manager_seed1",
        "TruthfulQA": "./workdir/output_dllm/qa/GSAI-ML/LLaDA-1.5/['truthful_qa', 'generation']/2025-06-30/07-41-34/ue_manager_seed1",
        "WMT14": "./workdir/output_dllm/nmt/GSAI-ML/LLaDA-1.5/['wmt14', 'fr-en']/2025-06-30/11-50-18/ue_manager_seed1",
        # "WMT14": "./workdir/output_dllm_dec25/nmt/GSAI-ML/LLaDA-1.5/['wmt14', 'fr-en']/2025-12-13/06-16-42/ue_manager_seed1",
        "WMT14": "./workdir/output_dllm_dec25/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2025-12-19/16-57-43/ue_manager_seed1",
        # "WMT19": "./workdir/output_dllm/nmt/GSAI-ML/LLaDA-1.5/['wmt19', 'de-en']/2025-06-30/11-50-18/ue_manager_seed1",
        "WMT19": "./workdir/output_dllm_dec25/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt19', 'deen_simple_instruct']/2025-12-23/18-17-25/ue_manager_seed1",
        # "TriviaQA": "./workdir/output_dllm/qa/GSAI-ML/LLaDA-1.5/['trivia_qa', 'rc.nocontext']/2025-06-30/20-09-50/ue_manager_seed1",
        "TriviaQA": "./workdir/output_dllm_dec25/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/triviaqa', 'simple_instruct']/2025-12-17/16-31-56/ue_manager_seed1",
        "MMLU": "./workdir/output_dllm/qa/GSAI-ML/LLaDA-1.5/['cais/mmlu', 'all']/2025-06-30/20-09-50/ue_manager_seed1",
    }
}

filename = "ue_manager_seed1"
dfs = {}

all_paths = []
for model in models:
    dfs[model] = {}
    for ds, ds_name in zip(datasets, datasets_names):
        task = ds.split("/")[0]
        ds = '/'.join(ds.split("/")[1:])
        path = f"{base_path}/{task}/{model}/{ds}/"
        final_path = None
        if model in paths.keys() and ds_name in paths[model].keys():
            final_path = paths[model][ds_name]
        else:
            continue
            for address, dirs, files in os.walk(path):
                for name in files:
                    full_path = os.path.join(address, name)
                    if full_path.endswith(filename):
                        final_path = copy.deepcopy(full_path)
        if final_path is not None:
            print(final_path)
            all_paths.append(final_path)
            if task == "ats":
                metric_names=['AlignScore']
            elif (ds_name in ["MMLU", "GSM8k"]):
                metric_names=["Accuracy"]
            elif task == "qa":
                metric_names=['AlignScore']
            elif task == "nmt":
                metric_names=['Comet']
            aggregated = True if ds_name in ["TruthfulQA", "TriviaQA"] else False
            df = get_table_short(final_path, dataset=ds_name, metric_names=metric_names, update_alignscore=False, aggregated=aggregated, return_score=False, upd=True, roc_auc=False, update_alignscore_sum=True, add_nll=True, mc_num=16)
            ind = np.argwhere(["P_i" in x or "MaximumSequenceProbability" in x for x in df[df.columns[-1:]].values.flatten()]).flatten()
            dfs[model][ds_name] = df.reset_index(drop=True)

In [ ]:
dfs_short = []
methods = None
model = models[0]

for key in dfs[model].keys():
    df_ = dfs[model][key]   
    if methods is None:
        methods = df_[df_.columns[-1]]
    df_ = df_.set_index(df_.columns[-1], drop=False).loc[methods]
    
    if len(dfs_short):
        dfs_short.append(df_[df_.columns[[0]]])
    else:
        dfs_short.append(df_[df_.columns[[1, 0]]])
dfs_short = pd.concat(dfs_short, axis=1).reset_index(drop=True)
dfs_short.loc[:, ('', "Mean")] = dfs_short[dfs_short.columns[1:]].mean(axis=1)
dfs_short.loc[:, ('', "Mean Rank")] = dfs_short[dfs_short.columns[1:-1]].apply(lambda x: rankdata(-x)).mean(axis=1).round(2)

In [ ]:
dfs_short.sort_values(dfs_short.columns[-1]).style.background_gradient(axis=0)

In [11]:
dfs_short.style.background_gradient(axis=0)

# New Methods

In [34]:
import os
import copy
from scipy.stats import rankdata

base_path = "./workdir/output_dllm_dec25_maskedll_final"

datasets = ['ats/xsum', 'ats/samsum', "ats/['cnn_dailymail', '3.0.0']", "nmt/['wmt14', 'fr-en']", "nmt/['wmt19', 'de-en']", "qa/keivalya/MedQuad-MedicalQnADataset", "qa/['truthful_qa', 'generation']", 'qa/coqa', 'qa/sciq', "qa/['trivia_qa', 'rc.nocontext']", "qa/['cais/mmlu', 'all']", "qa/['gsm8k', 'main']"]
datasets_names = ['XSUM', 'SamSum', 'CNN', "WMT14", "WMT19", "MedQUAD", "TruthfulQA", 'CoQA', 'SciQ', "TriviaQA", "MMLU", "GSM8k"]

models = ['GSAI-ML/LLaDA-1.5/']

paths = {
    "GSAI-ML/LLaDA-1.5/": {
        # "WMT14": "./workdir/output_dllm_dec25_maskedll/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2025-12-25/00-29-11/ue_manager_seed1",
        # "TriviaQA": "./workdir/output_dllm_dec25_maskedll/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/triviaqa', 'simple_instruct']/2025-12-24/20-02-47/ue_manager_seed1",
        # "SamSum": "./workdir/output_dllm_dec25_maskedll/ats/GSAI-ML/LLaDA-1.5/['LM-Polygraph/samsum', 'simple_instruct']/2025-12-25/07-56-31/ue_manager_seed1",
        "MMLU": "./workdir/output_dllm_dec25_maskedll_final/qa/{'path': 'GSAI-ML/LLaDA-1.5', 'ensemble': False, 'mc': False, 'mc_seeds': None, 'dropout_rate': None, 'type': 'DiffusionLLM', 'path_to_load_script': '/home/jovyan/artemvazhentsev/diffusion/llada/load_llada.py', 'load_model_args': {'device_map': 'auto'}, 'load_tokenizer_args': {}, 'device': 'cuda'}/['LM-Polygraph/mmlu', 'simple_instruct']/2026-01-06/18-33-53/ue_manager_seed1",
        # "WMT14": "./workdir/output_dllm_dec25_maskedll_final/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2026-01-01/13-03-50/ue_manager_seed1",
        "WMT14": "./workdir/output_dllm_dec25_maskedll_final/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2026-01-10/09-20-50/ue_manager_seed1",
        "WMT19": "./workdir/output_dllm_dec25_maskedll_final/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt19', 'deen_simple_instruct']/2026-01-09/16-12-44/ue_manager_seed1",
        # "WMT19": "./workdir/output_dllm_dec25_maskedll_final/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt19', 'deen_simple_instruct']/2025-12-30/22-21-46/ue_manager_seed1",
        "TriviaQA": "./workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/triviaqa', 'simple_instruct']/2025-12-31/22-48-30/ue_manager_seed1",
        "SamSum": "./workdir/output_dllm_dec25_maskedll_final/ats/GSAI-ML/LLaDA-1.5/['LM-Polygraph/samsum', 'simple_instruct']/2026-01-02/04-58-43/ue_manager_seed1",
        "GSM8k": "./workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/gsm8k', 'simple_instruct']/2026-01-04/00-37-05/ue_manager_seed1",
        "CoQA": "./workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-polygraph/coqa', 'simple_instruct']/2026-01-03/10-36-24/ue_manager_seed1",
        "TruthfulQA": "./workdir/output_dllm_dec25_maskedll_final/qa/{'path': 'GSAI-ML/LLaDA-1.5', 'ensemble': False, 'mc': False, 'mc_seeds': None, 'dropout_rate': None, 'type': 'DiffusionLLM', 'path_to_load_script': '/home/jovyan/artemvazhentsev/diffusion/llada/load_llada.py', 'load_model_args': {'device_map': 'auto'}, 'load_tokenizer_args': {}, 'device': 'cuda'}/['LM-Polygraph/truthfulqa', 'simple_instruct']/2026-01-02/21-34-36/ue_manager_seed1"
    }
}

filename = "ue_manager_seed1"
dfs = {}

all_paths = []
for model in models:
    dfs[model] = {}
    for ds, ds_name in zip(datasets, datasets_names):
        task = ds.split("/")[0]
        ds = '/'.join(ds.split("/")[1:])
        path = f"{base_path}/{task}/{model}/{ds}/"
        final_path = None
        if model in paths.keys() and ds_name in paths[model].keys():
            final_path = paths[model][ds_name]
        else:
            continue
            for address, dirs, files in os.walk(path):
                for name in files:
                    full_path = os.path.join(address, name)
                    if full_path.endswith(filename):
                        final_path = copy.deepcopy(full_path)
        if final_path is not None:
            print(final_path)
            all_paths.append(final_path)
            if task == "ats":
                metric_names=['AlignScore']
            elif (ds_name in ["MMLU", "GSM8k"]):
                metric_names=["Accuracy"]
            elif task == "qa":
                metric_names=['AlignScore']
            elif task == "nmt":
                metric_names=['Comet']
            aggregated = True if ds_name in ["TruthfulQA", "TriviaQA"] else False
            df = get_table_short(final_path, dataset=ds_name, metric_names=metric_names, update_alignscore=False, aggregated=aggregated, return_score=False, upd=False, roc_auc=False, update_alignscore_sum=False, add_nll=False, mc_num=16)
            ind = np.argwhere(["P_i" in x or "MaximumSequenceProbability" in x for x in df[df.columns[-1:]].values.flatten()]).flatten()
            dfs[model][ds_name] = df.reset_index(drop=True)

./workdir/output_dllm_dec25_maskedll_final/ats/GSAI-ML/LLaDA-1.5/['LM-Polygraph/samsum', 'simple_instruct']/2026-01-02/04-58-43/ue_manager_seed1


FileNotFoundError: [Errno 2] No such file or directory: "./workdir/output_dllm_dec25_maskedll_final/ats/GSAI-ML/LLaDA-1.5/['LM-Polygraph/samsum', 'simple_instruct']/2026-01-02/04-58-43/ue_manager_seed1"

In [ ]:
dfs_short = []
methods = None
model = models[0]

for key in dfs[model].keys():
    df_ = dfs[model][key]   
    if methods is None:
        methods = df_[df_.columns[-1]]
    df_ = df_.set_index(df_.columns[-1], drop=False).reindex(methods)
    
    if len(dfs_short):
        dfs_short.append(df_[df_.columns[[0]]])
    else:
        dfs_short.append(df_[df_.columns[[1, 0]]])
dfs_short = pd.concat(dfs_short, axis=1).reset_index(drop=True)
dfs_short.loc[:, ('', "Mean")] = dfs_short[dfs_short.columns[1:]].drop(columns=[('TruthfulQA', 'AlignScore')], errors='ignore').mean(axis=1)
dfs_short.loc[:, ('', "Mean Rank")] = dfs_short[dfs_short.columns[1:-1]].drop(columns=[('TruthfulQA', 'AlignScore')], errors='ignore').apply(lambda x: rankdata(-x)).mean(axis=1).round(2)

In [8]:
dfs_short.sort_values(dfs_short.columns[-2], ascending=False).style.background_gradient(axis=0)

In [39]:
dfs_short.sort_values(dfs_short.columns[-2], ascending=False).style.background_gradient(axis=0)

In [9]:
dfs_short.iloc[[30, 31]]

SamSum                WMT14     WMT19 TruthfulQA  \
                   Method AlignScore     Comet     Comet AlignScore   
30            MCMaskedNLL   0.176613  0.332584  0.390166  -0.134570   
31  MCMaskedNormalizedNLL   0.173182  0.433094  0.516595   0.285244   

         CoQA   TriviaQA      MMLU     GSM8k                      
   AlignScore AlignScore  Accuracy  Accuracy      Mean Mean Rank  
30   0.199879   0.206086 -0.091215  0.287897  0.214573     35.21  
31   0.163482   0.092666 -0.091215  0.044595  0.190343     35.93

In [6]:
import os
import copy
from scipy.stats import rankdata

base_path = "./workdir/output_dllm_dec25_maskedll_final"

datasets = ['ats/xsum', 'ats/samsum', "ats/['cnn_dailymail', '3.0.0']", "nmt/['wmt14', 'fr-en']", "nmt/['wmt19', 'de-en']", "qa/keivalya/MedQuad-MedicalQnADataset", "qa/['truthful_qa', 'generation']", 'qa/coqa', 'qa/sciq', "qa/['trivia_qa', 'rc.nocontext']", "qa/['cais/mmlu', 'all']", "qa/['gsm8k', 'main']"]
datasets_names = ['XSUM', 'SamSum', 'CNN', "WMT14", "WMT19", "MedQUAD", "TruthfulQA", 'CoQA', 'SciQ', "TriviaQA", "MMLU", "GSM8k"]


models = ['GSAI-ML/LLaDA-1.5/']

paths = {
    "GSAI-ML/LLaDA-1.5/": {
        # # "WMT14": "./workdir/output_dllm_dec25_maskedll/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2025-12-25/00-29-11/ue_manager_seed1",
        # # "TriviaQA": "./workdir/output_dllm_dec25_maskedll/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/triviaqa', 'simple_instruct']/2025-12-24/20-02-47/ue_manager_seed1",
        "SamSum": "../workdir/output_dllm_dec25_maskedll_final/ats/GSAI-ML/LLaDA-1.5/['LM-Polygraph/samsum', 'simple_instruct']/2026-01-13/22-40-23/ue_manager_seed1",
        # "SamSum": "./workdir/output_dllm_dec25_maskedll_final/ats/GSAI-ML/LLaDA-1.5/['LM-Polygraph/samsum', 'simple_instruct']/2026-01-09/11-52-05/ue_manager_seed1",
        "MMLU": "../workdir/output_dllm_dec25_maskedll_final/qa/{'path': 'GSAI-ML/LLaDA-1.5', 'ensemble': False, 'mc': False, 'mc_seeds': None, 'dropout_rate': None, 'type': 'DiffusionLLM', 'path_to_load_script': '/home/jovyan/artemvazhentsev/diffusion/llada/load_llada.py', 'load_model_args': {'device_map': 'auto'}, 'load_tokenizer_args': {}, 'device': 'cuda'}/['LM-Polygraph/mmlu', 'simple_instruct']/2026-01-14/10-33-48/ue_manager_seed1",
        # "WMT14": "./workdir/output_dllm_dec25_maskedll_final/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2026-01-08/23-54-26/ue_manager_seed1",
        "WMT14": "../workdir/output_dllm_dec25_maskedll_final/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2026-01-13/18-00-58/ue_manager_seed1",
        # "WMT19": "./workdir/output_dllm_dec25_maskedll_final/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt19', 'deen_simple_instruct']/2026-01-08/19-41-08/ue_manager_seed1",
        # "WMT19": "./workdir/output_dllm_dec25_maskedll_final/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt19', 'deen_simple_instruct']/2026-01-09/16-12-44/ue_manager_seed1",
        "WMT19": "../workdir/output_dllm_dec25_maskedll_final/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt19', 'deen_simple_instruct']/2026-01-13/15-25-51/ue_manager_seed1",
        "TriviaQA": "../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/triviaqa', 'simple_instruct']/2026-01-13/21-32-05/ue_manager_seed1",
        # # "SamSum": "./workdir/output_dllm_dec25_maskedll_final/ats/GSAI-ML/LLaDA-1.5/['LM-Polygraph/samsum', 'simple_instruct']/2026-01-02/04-58-43/ue_manager_seed1",
        "GSM8k": "../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/gsm8k', 'simple_instruct']/2026-01-13/23-57-27/ue_manager_seed1",
        # "CoQA": "./workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-polygraph/coqa', 'simple_instruct']/2026-01-11/02-41-07/ue_manager_seed1",
        "CoQA": "../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-polygraph/coqa', 'simple_instruct']/2026-01-13/20-31-35/ue_manager_seed1",
        "TruthfulQA": "../workdir/output_dllm_dec25_maskedll_final/qa/{'path': 'GSAI-ML/LLaDA-1.5', 'ensemble': False, 'mc': False, 'mc_seeds': None, 'dropout_rate': None, 'type': 'DiffusionLLM', 'path_to_load_script': '/home/jovyan/artemvazhentsev/diffusion/llada/load_llada.py', 'load_model_args': {'device_map': 'auto'}, 'load_tokenizer_args': {}, 'device': 'cuda'}/['LM-Polygraph/truthfulqa', 'simple_instruct']/2026-01-14/02-59-12/ue_manager_seed1"
    }
}

filename = "ue_manager_seed1"
dfs = {}

all_paths = []
for model in models:
    dfs[model] = {}
    for ds, ds_name in zip(datasets, datasets_names):
        task = ds.split("/")[0]
        ds = '/'.join(ds.split("/")[1:])
        path = f"{base_path}/{task}/{model}/{ds}/"
        final_path = None
        if model in paths.keys() and ds_name in paths[model].keys():
            final_path = paths[model][ds_name]
        else:
            continue
            for address, dirs, files in os.walk(path):
                for name in files:
                    full_path = os.path.join(address, name)
                    if full_path.endswith(filename):
                        final_path = copy.deepcopy(full_path)
        if final_path is not None:
            print(final_path)
            all_paths.append(final_path)
            if task == "ats":
                metric_names=['AlignScore']
            elif (ds_name in ["MMLU", "GSM8k"]):
                metric_names=["Accuracy"]
            elif task == "qa":
                metric_names=['AlignScore']
            elif task == "nmt":
                metric_names=['Comet']
            aggregated = True if ds_name in ["TruthfulQA", "TriviaQA"] else False
            df = get_table_short(final_path, dataset=ds_name, metric_names=metric_names, update_alignscore=False, aggregated=aggregated, return_score=False, upd=False, roc_auc=False, update_alignscore_sum=False, add_nll=False, mc_num=16)
            ind = np.argwhere(["P_i" in x or "MaximumSequenceProbability" in x for x in df[df.columns[-1:]].values.flatten()]).flatten()
            dfs[model][ds_name] = df.reset_index(drop=True)

../workdir/output_dllm_dec25_maskedll_final/ats/GSAI-ML/LLaDA-1.5/['LM-Polygraph/samsum', 'simple_instruct']/2026-01-13/22-40-23/ue_manager_seed1
AlignScore:  0.8963388724284421
../workdir/output_dllm_dec25_maskedll_final/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2026-01-13/18-00-58/ue_manager_seed1
Comet:  0.8576546686068177
../workdir/output_dllm_dec25_maskedll_final/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt19', 'deen_simple_instruct']/2026-01-13/15-25-51/ue_manager_seed1
Comet:  0.8490201873853803
../workdir/output_dllm_dec25_maskedll_final/qa/{'path': 'GSAI-ML/LLaDA-1.5', 'ensemble': False, 'mc': False, 'mc_seeds': None, 'dropout_rate': None, 'type': 'DiffusionLLM', 'path_to_load_script': '/home/jovyan/artemvazhentsev/diffusion/llada/load_llada.py', 'load_model_args': {'device_map': 'auto'}, 'load_tokenizer_args': {}, 'device': 'cuda'}/['LM-Polygraph/truthfulqa', 'simple_instruct']/2026-01-14/02-59-12/ue_manager_seed1
AlignScore:  0.4354578974746922
../wor

In [7]:
dfs_short = []
methods = None
model = models[0]

for key in dfs[model].keys():
    df_ = dfs[model][key]   
    if methods is None:
        methods = df_[df_.columns[-1]]
    df_ = df_.set_index(df_.columns[-1], drop=False).reindex(methods)
    
    if len(dfs_short):
        dfs_short.append(df_[df_.columns[[0]]])
    else:
        dfs_short.append(df_[df_.columns[[1, 0]]])
dfs_short = pd.concat(dfs_short, axis=1).reset_index(drop=True)
dfs_short.loc[:, ('', "Mean")] = dfs_short[dfs_short.columns[1:]].mean(axis=1)
dfs_short.loc[:, ('', "Mean Rank")] = dfs_short[dfs_short.columns[1:-1]].apply(lambda x: rankdata(-x)).mean(axis=1).round(2)

In [11]:
dfs_short.sort_values(dfs_short.columns[-2], ascending=False).style.background_gradient(axis=0)

In [8]:
dfs_short.sort_values(dfs_short.columns[-2], ascending=False).style.background_gradient(axis=0)

In [8]:
import os
import copy
from scipy.stats import rankdata

base_path = "./workdir/output_dllm_dec25_maskedll_final"

datasets = ['ats/xsum', 'ats/samsum', "ats/['cnn_dailymail', '3.0.0']", "nmt/['wmt14', 'fr-en']", "nmt/['wmt19', 'de-en']", "qa/keivalya/MedQuad-MedicalQnADataset", "qa/['truthful_qa', 'generation']", 'qa/coqa', 'qa/sciq', "qa/['trivia_qa', 'rc.nocontext']", "qa/['cais/mmlu', 'all']", "qa/['gsm8k', 'main']"]
datasets_names = ['XSUM', 'SamSum', 'CNN', "WMT14", "WMT19", "MedQUAD", "TruthfulQA", 'CoQA', 'SciQ', "TriviaQA", "MMLU", "GSM8k"]


models = ['GSAI-ML/LLaDA-1.5/']

paths = {
    "GSAI-ML/LLaDA-1.5/": {
        # # # "WMT14": "./workdir/output_dllm_dec25_maskedll/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2025-12-25/00-29-11/ue_manager_seed1",
        # # # "TriviaQA": "./workdir/output_dllm_dec25_maskedll/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/triviaqa', 'simple_instruct']/2025-12-24/20-02-47/ue_manager_seed1",
        "SamSum": "../workdir/output_dllm_dec25_maskedll_final/ats/GSAI-ML/LLaDA-1.5/['LM-Polygraph/samsum', 'simple_instruct']/2026-03-03/19-09-44/ue_manager_seed1",
        # # "SamSum": "./workdir/output_dllm_dec25_maskedll_final/ats/GSAI-ML/LLaDA-1.5/['LM-Polygraph/samsum', 'simple_instruct']/2026-01-09/11-52-05/ue_manager_seed1",
        "MMLU": "../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/mmlu', 'simple_instruct']/2026-03-03/17-53-22/ue_manager_seed1",
        # # "WMT14": "./workdir/output_dllm_dec25_maskedll_final/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2026-01-08/23-54-26/ue_manager_seed1",
        "WMT14": "../workdir/output_dllm_dec25_maskedll_final/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2026-03-03/19-45-16/ue_manager_seed1",
        # # "WMT19": "./workdir/output_dllm_dec25_maskedll_final/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt19', 'deen_simple_instruct']/2026-01-08/19-41-08/ue_manager_seed1",
        # # "WMT19": "./workdir/output_dllm_dec25_maskedll_final/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt19', 'deen_simple_instruct']/2026-01-09/16-12-44/ue_manager_seed1",
        "WMT19": "../workdir/output_dllm_dec25_maskedll_final/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt19', 'deen_simple_instruct']/2026-03-03/19-45-27/ue_manager_seed1",
        "TriviaQA": "../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/triviaqa', 'simple_instruct']/2026-03-02/22-36-59/ue_manager_seed1",
        # # # "SamSum": "./workdir/output_dllm_dec25_maskedll_final/ats/GSAI-ML/LLaDA-1.5/['LM-Polygraph/samsum', 'simple_instruct']/2026-01-02/04-58-43/ue_manager_seed1",
        "GSM8k": "../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/gsm8k', 'simple_instruct']/2026-03-03/18-24-49/ue_manager_seed1",
        # # "CoQA": "./workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-polygraph/coqa', 'simple_instruct']/2026-01-11/02-41-07/ue_manager_seed1",
        # "CoQA": "../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-polygraph/coqa', 'simple_instruct']/2026-02-06/13-45-59/ue_manager_seed1",
        "CoQA": "../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-polygraph/coqa', 'simple_instruct']/2026-03-02/20-54-34/ue_manager_seed1",
        "TruthfulQA": "../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/truthfulqa', 'simple_instruct']/2026-03-03/00-22-34/ue_manager_seed1"
    }
}

filename = "ue_manager_seed1"
dfs = {}

all_paths = []
for model in models:
    dfs[model] = {}
    for ds, ds_name in zip(datasets, datasets_names):
        task = ds.split("/")[0]
        ds = '/'.join(ds.split("/")[1:])
        path = f"{base_path}/{task}/{model}/{ds}/"
        final_path = None
        if model in paths.keys() and ds_name in paths[model].keys():
            final_path = paths[model][ds_name]
        else:
            continue
            for address, dirs, files in os.walk(path):
                for name in files:
                    full_path = os.path.join(address, name)
                    if full_path.endswith(filename):
                        final_path = copy.deepcopy(full_path)
        if final_path is not None:
            print(final_path)
            all_paths.append(final_path)
            if task == "ats":
                metric_names=['AlignScore']
            elif (ds_name in ["MMLU", "GSM8k"]):
                metric_names=["Accuracy"]
            elif task == "qa":
                metric_names=['AlignScore']
            elif task == "nmt":
                metric_names=['Comet']
            aggregated = True if ds_name in ["TruthfulQA", "TriviaQA"] else False
            df = get_table_short(final_path, dataset=ds_name, metric_names=metric_names, update_alignscore=False, aggregated=aggregated, return_score=False, upd=False, roc_auc=False, update_alignscore_sum=False, add_nll=False, mc_num=16)
            ind = np.argwhere(["P_i" in x or "MaximumSequenceProbability" in x for x in df[df.columns[-1:]].values.flatten()]).flatten()
            dfs[model][ds_name] = df.reset_index(drop=True)

../workdir/output_dllm_dec25_maskedll_final/ats/GSAI-ML/LLaDA-1.5/['LM-Polygraph/samsum', 'simple_instruct']/2026-03-03/19-09-44/ue_manager_seed1
AlignScore:  0.8985490752978226
../workdir/output_dllm_dec25_maskedll_final/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2026-03-03/19-45-16/ue_manager_seed1
Comet:  0.8578273457363248
../workdir/output_dllm_dec25_maskedll_final/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt19', 'deen_simple_instruct']/2026-03-03/19-45-27/ue_manager_seed1
Comet:  0.8497599904537201
../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/truthfulqa', 'simple_instruct']/2026-03-03/00-22-34/ue_manager_seed1
AlignScore:  0.43344209848283566
../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-polygraph/coqa', 'simple_instruct']/2026-03-02/20-54-34/ue_manager_seed1
AlignScore:  0.736534931716822
../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/triviaqa', 'simple_instruct']/20

In [9]:
dfs_short = []
methods = None
model = models[0]

for key in dfs[model].keys():
    df_ = dfs[model][key]   
    if methods is None:
        methods = df_[df_.columns[-1]]
    df_ = df_.set_index(df_.columns[-1], drop=False).reindex(methods)
    
    if len(dfs_short):
        dfs_short.append(df_[df_.columns[[0]]])
    else:
        dfs_short.append(df_[df_.columns[[1, 0]]])
dfs_short = pd.concat(dfs_short, axis=1).reset_index(drop=True)
dfs_short.loc[:, ('', "Mean")] = dfs_short[dfs_short.columns[1:]].mean(axis=1)
dfs_short.loc[:, ('', "Mean Rank")] = dfs_short[dfs_short.columns[1:-1]].apply(lambda x: rankdata(-x)).mean(axis=1).round(2)

In [10]:
dfs_short.sort_values(dfs_short.columns[-2], ascending=False).style.background_gradient(axis=0)

In [7]:
import os
import copy
from scipy.stats import rankdata

base_path = "./workdir/output_dllm_dec25_maskedll_final"

datasets = ['ats/xsum', 'ats/samsum', "ats/['cnn_dailymail', '3.0.0']", "nmt/['wmt14', 'fr-en']", "nmt/['wmt19', 'de-en']", "qa/keivalya/MedQuad-MedicalQnADataset", "qa/['truthful_qa', 'generation']", 'qa/coqa', 'qa/sciq', "qa/['trivia_qa', 'rc.nocontext']", "qa/['cais/mmlu', 'all']", "qa/['gsm8k', 'main']"]
datasets_names = ['XSUM', 'SamSum', 'CNN', "WMT14", "WMT19", "MedQUAD", "TruthfulQA", 'CoQA', 'SciQ', "TriviaQA", "MMLU", "GSM8k"]


models = ['GSAI-ML/LLaDA-1.5/']

paths = {
    "GSAI-ML/LLaDA-1.5/": {
        # # # "WMT14": "./workdir/output_dllm_dec25_maskedll/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2025-12-25/00-29-11/ue_manager_seed1",
        # # # "TriviaQA": "./workdir/output_dllm_dec25_maskedll/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/triviaqa', 'simple_instruct']/2025-12-24/20-02-47/ue_manager_seed1",
        "SamSum": "../workdir/output_dllm_dec25_maskedll_final/ats/GSAI-ML/LLaDA-1.5/['LM-Polygraph/samsum', 'simple_instruct']/2026-03-05/16-24-42/ue_manager_seed1",
        # # "SamSum": "./workdir/output_dllm_dec25_maskedll_final/ats/GSAI-ML/LLaDA-1.5/['LM-Polygraph/samsum', 'simple_instruct']/2026-01-09/11-52-05/ue_manager_seed1",
        # "MMLU": "../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/mmlu', 'simple_instruct']/2026-03-03/17-53-22/ue_manager_seed1",
        # # "WMT14": "./workdir/output_dllm_dec25_maskedll_final/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2026-01-08/23-54-26/ue_manager_seed1",
        "WMT14": "../workdir/output_dllm_dec25_maskedll_final/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2026-03-05/15-16-31/ue_manager_seed1",
        # # "WMT19": "./workdir/output_dllm_dec25_maskedll_final/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt19', 'deen_simple_instruct']/2026-01-08/19-41-08/ue_manager_seed1",
        # # "WMT19": "./workdir/output_dllm_dec25_maskedll_final/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt19', 'deen_simple_instruct']/2026-01-09/16-12-44/ue_manager_seed1",
        "WMT19": "../workdir/output_dllm_dec25_maskedll_final/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt19', 'deen_simple_instruct']/2026-03-05/15-16-30/ue_manager_seed1",
        # "TriviaQA": "../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/triviaqa', 'simple_instruct']/2026-03-02/22-36-59/ue_manager_seed1",
        # # # "SamSum": "./workdir/output_dllm_dec25_maskedll_final/ats/GSAI-ML/LLaDA-1.5/['LM-Polygraph/samsum', 'simple_instruct']/2026-01-02/04-58-43/ue_manager_seed1",
        "GSM8k": "../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/gsm8k', 'simple_instruct']/2026-03-05/15-17-42/ue_manager_seed1",
        # # "CoQA": "./workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-polygraph/coqa', 'simple_instruct']/2026-01-11/02-41-07/ue_manager_seed1",
        # "CoQA": "../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-polygraph/coqa', 'simple_instruct']/2026-02-06/13-45-59/ue_manager_seed1",
        "CoQA": "../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-polygraph/coqa', 'simple_instruct']/2026-03-06/16-40-40/ue_manager_seed1",
        "TruthfulQA": "../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/truthfulqa', 'simple_instruct']/2026-03-05/15-16-20/ue_manager_seed1",
        "XSUM": "../workdir/output_dllm_dec25_maskedll_final/ats/GSAI-ML/LLaDA-1.5/['LM-Polygraph/xsum', 'simple_instruct']/2026-03-06/12-26-19/ue_manager_seed1"
    }
}

filename = "ue_manager_seed1"
dfs = {}

all_paths = []
for model in models:
    dfs[model] = {}
    for ds, ds_name in zip(datasets, datasets_names):
        task = ds.split("/")[0]
        ds = '/'.join(ds.split("/")[1:])
        path = f"{base_path}/{task}/{model}/{ds}/"
        final_path = None
        if model in paths.keys() and ds_name in paths[model].keys():
            final_path = paths[model][ds_name]
        else:
            continue
            for address, dirs, files in os.walk(path):
                for name in files:
                    full_path = os.path.join(address, name)
                    if full_path.endswith(filename):
                        final_path = copy.deepcopy(full_path)
        if final_path is not None:
            print(final_path)
            all_paths.append(final_path)
            if task == "ats":
                metric_names=['AlignScore']
            elif (ds_name in ["MMLU", "GSM8k"]):
                metric_names=["Accuracy"]
            elif task == "qa":
                metric_names=['AlignScore']
            elif task == "nmt":
                metric_names=['Comet']
            aggregated = True if ds_name in ["TruthfulQA", "TriviaQA"] else False
            df = get_table_short(final_path, dataset=ds_name, metric_names=metric_names, update_alignscore=False, aggregated=aggregated, return_score=False, upd=False, roc_auc=False, update_alignscore_sum=False, add_nll=False, mc_num=16)
            ind = np.argwhere(["P_i" in x or "MaximumSequenceProbability" in x for x in df[df.columns[-1:]].values.flatten()]).flatten()
            dfs[model][ds_name] = df.reset_index(drop=True)

../workdir/output_dllm_dec25_maskedll_final/ats/GSAI-ML/LLaDA-1.5/['LM-Polygraph/xsum', 'simple_instruct']/2026-03-06/12-26-19/ue_manager_seed1
AlignScore:  0.8466164778063249
../workdir/output_dllm_dec25_maskedll_final/ats/GSAI-ML/LLaDA-1.5/['LM-Polygraph/samsum', 'simple_instruct']/2026-03-05/16-24-42/ue_manager_seed1
AlignScore:  0.8985490752978226
../workdir/output_dllm_dec25_maskedll_final/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2026-03-05/15-16-31/ue_manager_seed1
Comet:  0.8578273457363248
../workdir/output_dllm_dec25_maskedll_final/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt19', 'deen_simple_instruct']/2026-03-05/15-16-30/ue_manager_seed1
Comet:  0.8497599904537201
../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/truthfulqa', 'simple_instruct']/2026-03-05/15-16-20/ue_manager_seed1
AlignScore:  0.43344209848283566
../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-polygraph/coqa', 'simple_instruct']/2026

In [14]:
dfs_short = []
methods = None
model = models[0]

for key in dfs[model].keys():
    df_ = dfs[model][key]   
    if methods is None:
        methods = df_[df_.columns[-1]]
        # methods = [m for m in methods if "AvgDissimilarity" in m]
    df_ = df_.set_index(df_.columns[-1], drop=False).reindex(methods)
    
    if len(dfs_short):
        dfs_short.append(df_[df_.columns[[0]]])
    else:
        dfs_short.append(df_[df_.columns[[1, 0]]])
dfs_short = pd.concat(dfs_short, axis=1).reset_index(drop=True)
dfs_short.loc[:, ('', "Mean")] = dfs_short[dfs_short.columns[1:]].mean(axis=1)
dfs_short.loc[:, ('', "Mean Rank")] = dfs_short[dfs_short.columns[1:-1]].apply(lambda x: rankdata(-x)).mean(axis=1).round(2)

In [15]:
dfs_short.sort_values(dfs_short.columns[-2], ascending=False).style.background_gradient(axis=0)

In [13]:
dfs_short.sort_values(dfs_short.columns[-2], ascending=False).style.background_gradient(axis=0)

In [81]:
import os
import copy
from scipy.stats import rankdata

base_path = "./workdir/output_dllm_dec25_maskedll_final"

datasets = ['ats/xsum', 'ats/samsum', "ats/['cnn_dailymail', '3.0.0']", "nmt/['wmt14', 'fr-en']", "nmt/['wmt19', 'de-en']", "qa/keivalya/MedQuad-MedicalQnADataset", "qa/['truthful_qa', 'generation']", 'qa/coqa', 'qa/sciq', "qa/['trivia_qa', 'rc.nocontext']", "qa/['cais/mmlu', 'all']", "qa/['gsm8k', 'main']"]
datasets_names = ['XSUM', 'SamSum', 'CNN', "WMT14", "WMT19", "MedQUAD", "TruthfulQA", 'CoQA', 'SciQ', "TriviaQA", "MMLU", "GSM8k"]


models = ['GSAI-ML/LLaDA-1.5/']

paths = {
    "GSAI-ML/LLaDA-1.5/": {
        # # # "WMT14": "./workdir/output_dllm_dec25_maskedll/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2025-12-25/00-29-11/ue_manager_seed1",
        # # # # "TriviaQA": "./workdir/output_dllm_dec25_maskedll/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/triviaqa', 'simple_instruct']/2025-12-24/20-02-47/ue_manager_seed1",
        # "SamSum": "../workdir/output_dllm_dec25_maskedll_final/ats/GSAI-ML/LLaDA-1.5/['LM-Polygraph/samsum', 'simple_instruct']/2026-03-05/16-24-42/ue_manager_seed1",
        # # # "SamSum": "./workdir/output_dllm_dec25_maskedll_final/ats/GSAI-ML/LLaDA-1.5/['LM-Polygraph/samsum', 'simple_instruct']/2026-01-09/11-52-05/ue_manager_seed1",
        # # "MMLU": "../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/mmlu', 'simple_instruct']/2026-03-03/17-53-22/ue_manager_seed1",
        # # # "WMT14": "./workdir/output_dllm_dec25_maskedll_final/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2026-01-08/23-54-26/ue_manager_seed1",
        # "WMT14": "../workdir/output_dllm_dec25_maskedll_final/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2026-03-05/15-16-31/ue_manager_seed1",
        # # # "WMT19": "./workdir/output_dllm_dec25_maskedll_final/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt19', 'deen_simple_instruct']/2026-01-08/19-41-08/ue_manager_seed1",
        # # # "WMT19": "./workdir/output_dllm_dec25_maskedll_final/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt19', 'deen_simple_instruct']/2026-01-09/16-12-44/ue_manager_seed1",
        # "WMT19": "../workdir/output_dllm_dec25_maskedll_final/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt19', 'deen_simple_instruct']/2026-03-16/18-21-33/ue_manager_seed1",
        # # "TriviaQA": "../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/triviaqa', 'simple_instruct']/2026-03-02/22-36-59/ue_manager_seed1",
        # # # # "SamSum": "./workdir/output_dllm_dec25_maskedll_final/ats/GSAI-ML/LLaDA-1.5/['LM-Polygraph/samsum', 'simple_instruct']/2026-01-02/04-58-43/ue_manager_seed1",
        "GSM8k": "../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/gsm8k', 'simple_instruct']/2026-03-16/17-28-19/ue_manager_seed1",
        # # # "CoQA": "./workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-polygraph/coqa', 'simple_instruct']/2026-01-11/02-41-07/ue_manager_seed1",
        # # "CoQA": "../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-polygraph/coqa', 'simple_instruct']/2026-02-06/13-45-59/ue_manager_seed1",
        # "CoQA": "../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-polygraph/coqa', 'simple_instruct']/2026-03-06/16-40-40/ue_manager_seed1",
        # "TruthfulQA": "../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/truthfulqa', 'simple_instruct']/2026-03-05/15-16-20/ue_manager_seed1",
        # "XSUM": "../workdir/output_dllm_dec25_maskedll_final/ats/GSAI-ML/LLaDA-1.5/['LM-Polygraph/xsum', 'simple_instruct']/2026-03-06/12-26-19/ue_manager_seed1"
    }
}

paths = {
    "GSAI-ML/LLaDA-1.5/": {
        # "SamSum": "../workdir/output_dllm_dec25_maskedll_final/ats/GSAI-ML/LLaDA-1.5/['LM-Polygraph/samsum', 'simple_instruct']/2026-03-17/16-22-37/ue_manager_seed1",
        # "XSUM": "../workdir/output_dllm_dec25_maskedll_final/ats/GSAI-ML/LLaDA-1.5/['LM-Polygraph/xsum', 'simple_instruct']/2026-03-17/21-56-53/ue_manager_seed1",
        # "WMT14": "../workdir/output_dllm_dec25_maskedll_final/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2026-03-17/18-04-24/ue_manager_seed1",
        # "WMT19": "../workdir/output_dllm_dec25_maskedll_final/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt19', 'deen_simple_instruct']/2026-03-17/18-26-48/ue_manager_seed1",
        # "CoQA": "../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-polygraph/coqa', 'simple_instruct']/2026-03-20/12-16-45/ue_manager_seed1",
        # "GSM8k": "../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/gsm8k', 'simple_instruct']/2026-03-18/23-30-23/ue_manager_seed1",
        # "MMLU": "../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/mmlu', 'simple_instruct']/2026-03-03/17-53-22/ue_manager_seed1",
        # "TriviaQA": "../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/triviaqa', 'simple_instruct']/2026-03-09/13-41-00/ue_manager_seed1",
        # "TruthfulQA": "../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/truthfulqa', 'simple_instruct']/2026-03-17/13-56-33/ue_manager_seed1",
    }
}

paths = {
    "GSAI-ML/LLaDA-1.5/": {
        "SamSum": "../workdir/output_dllm_dec25_maskedll_final/ats/GSAI-ML/LLaDA-1.5/['LM-Polygraph/samsum', 'simple_instruct']/2026-03-23/22-16-35/ue_manager_seed1",
        # "XSUM": "../workdir/output_dllm_dec25_maskedll_final/ats/GSAI-ML/LLaDA-1.5/['LM-Polygraph/xsum', 'simple_instruct']/2026-03-24/06-20-55/ue_manager_seed1",
        "WMT14": "../workdir/output_dllm_dec25_maskedll_final/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2026-03-23/22-18-33/ue_manager_seed1",
        "WMT19": "../workdir/output_dllm_dec25_maskedll_final/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt19', 'deen_simple_instruct']/2026-03-24/01-19-28/ue_manager_seed1",
        "CoQA": "../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-polygraph/coqa', 'simple_instruct']/2026-03-23/21-24-22/ue_manager_seed1",
        # "CoQA": "../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-polygraph/coqa', 'simple_instruct']/2026-03-24/13-55-20/ue_manager_seed1",
        "GSM8k": "../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/gsm8k', 'simple_instruct']/2026-03-23/17-00-10/ue_manager_seed1",
        # "GSM8k": "../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/gsm8k', 'simple_instruct']/2026-03-24/13-37-59/ue_manager_seed1",
        "MMLU": "../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/mmlu', 'simple_instruct']/2026-03-24/00-14-07/ue_manager_seed1",
        "TriviaQA": "../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/triviaqa', 'simple_instruct']/2026-03-24/01-29-15/ue_manager_seed1",
        # "TruthfulQA": "../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/truthfulqa', 'simple_instruct']/2026-03-24/03-51-34/ue_manager_seed1",
    }
}

paths = {
    "GSAI-ML/LLaDA-1.5/": {
        "SamSum": "../workdir/output_dllm_dec25_maskedll_final/ats/GSAI-ML/LLaDA-1.5/['LM-Polygraph/samsum', 'simple_instruct']/2026-03-25/01-04-55/ue_manager_seed1",
        # "XSUM": "../workdir/output_dllm_dec25_maskedll_final/ats/GSAI-ML/LLaDA-1.5/['LM-Polygraph/xsum', 'simple_instruct']/2026-03-25/16-52-38/ue_manager_seed1",
        "WMT14": "../workdir/output_dllm_dec25_maskedll_final/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2026-03-25/04-09-53/ue_manager_seed1",
        "WMT19": "../workdir/output_dllm_dec25_maskedll_final/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt19', 'deen_simple_instruct']/2026-03-25/08-14-48/ue_manager_seed1",
        "CoQA": "../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-polygraph/coqa', 'simple_instruct']/2026-03-25/00-53-35/ue_manager_seed1",
        # "GSM8k": "../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/gsm8k', 'simple_instruct']/2026-03-24/14-28-05/ue_manager_seed1",
        "MMLU": "../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/mmlu', 'simple_instruct']/2026-03-25/15-16-53/ue_manager_seed1",
        "TriviaQA": "../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/triviaqa', 'simple_instruct']/2026-03-25/05-21-47/ue_manager_seed1",
        # "TruthfulQA": "../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/truthfulqa', 'simple_instruct']/2026-03-25/07-52-54/ue_manager_seed1",
        
    }
}




filename = "ue_manager_seed1"
dfs = {}

all_paths = []
for model in models:
    dfs[model] = {}
    for ds, ds_name in zip(datasets, datasets_names):
        task = ds.split("/")[0]
        ds = '/'.join(ds.split("/")[1:])
        path = f"{base_path}/{task}/{model}/{ds}/"
        final_path = None
        if model in paths.keys() and ds_name in paths[model].keys():
            final_path = paths[model][ds_name]
        else:
            continue
            for address, dirs, files in os.walk(path):
                for name in files:
                    full_path = os.path.join(address, name)
                    if full_path.endswith(filename):
                        final_path = copy.deepcopy(full_path)
        if final_path is not None:
            print(final_path)
            all_paths.append(final_path)
            if task == "ats":
                metric_names=['AlignScore']
            elif (ds_name in ["MMLU", "GSM8k"]):
                metric_names=["Accuracy"]
            elif task == "qa":
                metric_names=['AlignScore']
            elif task == "nmt":
                metric_names=['Comet']
            aggregated = True if ds_name in ["TruthfulQA", "TriviaQA"] else False
            df = get_table_short(final_path, dataset=ds_name, metric_names=metric_names, update_alignscore=False, aggregated=aggregated, return_score=False, upd=False, roc_auc=False, update_alignscore_sum=False, add_nll=False, mc_num=16)
            ind = np.argwhere(["P_i" in x or "MaximumSequenceProbability" in x for x in df[df.columns[-1:]].values.flatten()]).flatten()
            dfs[model][ds_name] = df.reset_index(drop=True)

../workdir/output_dllm_dec25_maskedll_final/ats/GSAI-ML/LLaDA-1.5/['LM-Polygraph/samsum', 'simple_instruct']/2026-03-25/01-04-55/ue_manager_seed1
AlignScore:  0.8985490752978226
../workdir/output_dllm_dec25_maskedll_final/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2026-03-25/04-09-53/ue_manager_seed1
Comet:  0.8578273457363248
../workdir/output_dllm_dec25_maskedll_final/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt19', 'deen_simple_instruct']/2026-03-25/08-14-48/ue_manager_seed1
Comet:  0.8497599904537201
../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-polygraph/coqa', 'simple_instruct']/2026-03-25/00-53-35/ue_manager_seed1
AlignScore:  0.736534931716822
../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/triviaqa', 'simple_instruct']/2026-03-25/05-21-47/ue_manager_seed1
AlignScore:  0.45934441404898096
../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/mmlu', 'simple_instruct']/2026-03-

In [82]:
dfs_short_new = []
methods = None
model = models[0]

for key in dfs[model].keys():
    df_ = dfs[model][key]   
    if methods is None:
        methods = df_[df_.columns[-1]]
        # methods = [m for m in methods if "FullTrace" in m]
        # methods = [m for m in methods if "InverseProgress" not in m]
    df_ = df_.set_index(df_.columns[-1], drop=False).reindex(methods)
    
    if len(dfs_short_new):
        dfs_short_new.append(df_[df_.columns[[0]]])
    else:
        dfs_short_new.append(df_[df_.columns[[1, 0]]])
dfs_short_new = pd.concat(dfs_short_new, axis=1).reset_index(drop=True)
# dfs_short.loc[:, ('', "Mean")] = dfs_short[dfs_short.columns[1:]].mean(axis=1)
# dfs_short.loc[:, ('', "Mean Rank")] = dfs_short[dfs_short.columns[1:-1]].apply(lambda x: rankdata(-x)).mean(axis=1).round(2)

In [83]:
dfs_short_new.sort_values(dfs_short_new.columns[-2], ascending=False).style.background_gradient(axis=0)#.iloc[[165, 199, 147, 67, 79, 85, 295, 257, 288, 287, 109]]

In [84]:
dfs_short_new.iloc[[199, 165, 151, 273, 297, 285, 153]].sort_values(dfs_short.columns[-2], ascending=False).style.background_gradient(axis=0)

In [85]:
dfs_short = []
methods = None
model = models[0]

for key in dfs[model].keys():
    if key == "TruthfulQA" or key == "MMLU":
        continue
    df_ = dfs[model][key]   
    if methods is None:
        methods = df_[df_.columns[-1]]
        # methods = [m for m in methods if "CrossEncoder" in m]
    df_ = df_.set_index(df_.columns[-1], drop=False).reindex(methods)
    
    if len(dfs_short):
        dfs_short.append(df_[df_.columns[[0]]])
    else:
        dfs_short.append(df_[df_.columns[[1, 0]]])
dfs_short = pd.concat(dfs_short, axis=1).reset_index(drop=True)
dfs_short.loc[:, ('', "Mean")] = dfs_short[dfs_short.columns[1:]].mean(axis=1)
dfs_short.loc[:, ('', "Mean Rank")] = dfs_short[dfs_short.columns[1:-1]].apply(lambda x: rankdata(-x)).mean(axis=1).round(2)

dfs_short.sort_values(dfs_short.columns[-2], ascending=False).style.background_gradient(axis=0)

In [86]:
import os
import copy
from scipy.stats import rankdata

base_path = "./workdir/output_dllm_dec25_maskedll_final"

datasets = ['ats/xsum', 'ats/samsum', "ats/['cnn_dailymail', '3.0.0']", "nmt/['wmt14', 'fr-en']", "nmt/['wmt19', 'de-en']", "qa/keivalya/MedQuad-MedicalQnADataset", "qa/['truthful_qa', 'generation']", 'qa/coqa', 'qa/sciq', "qa/['trivia_qa', 'rc.nocontext']", "qa/['cais/mmlu', 'all']", "qa/['gsm8k', 'main']"]
datasets_names = ['XSUM', 'SamSum', 'CNN', "WMT14", "WMT19", "MedQUAD", "TruthfulQA", 'CoQA', 'SciQ', "TriviaQA", "MMLU", "GSM8k"]


models = ['GSAI-ML/LLaDA-1.5/']

paths = {
    "GSAI-ML/LLaDA-1.5/": {
        "SamSum": "../workdir/dllm_baselines/ats/GSAI-ML/LLaDA-1.5/['LM-Polygraph/samsum', 'simple_instruct']/2026-03-27/01-38-57/ue_manager_seed1",
        "WMT14": "../workdir/dllm_baselines/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2026-03-27/10-12-46/ue_manager_seed1",
        "WMT19": "../workdir/dllm_baselines/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt19', 'deen_simple_instruct']/2026-03-27/12-39-25/ue_manager_seed1",
        "TriviaQA": "../workdir/dllm_baselines/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/triviaqa', 'simple_instruct']/2026-03-26/23-37-58/ue_manager_seed1",
        "CoQA": "../workdir/dllm_baselines/qa/GSAI-ML/LLaDA-1.5/['LM-polygraph/coqa', 'simple_instruct']/2026-03-31/10-25-58/ue_manager_seed1",
        # "TruthfulQA": "../workdir/dllm_baselines/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/truthfulqa', 'simple_instruct']/2026-04-07/06-40-05/ue_manager_seed1",
        "MMLU": "../workdir/dllm_baselines/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/mmlu', 'simple_instruct']/2026-04-07/23-36-27/ue_manager_seed1",
    }
}


filename = "ue_manager_seed1"
dfs = {}

all_paths = []
for model in models:
    dfs[model] = {}
    for ds, ds_name in zip(datasets, datasets_names):
        task = ds.split("/")[0]
        ds = '/'.join(ds.split("/")[1:])
        path = f"{base_path}/{task}/{model}/{ds}/"
        final_path = None
        if model in paths.keys() and ds_name in paths[model].keys():
            final_path = paths[model][ds_name]
        else:
            continue
            for address, dirs, files in os.walk(path):
                for name in files:
                    full_path = os.path.join(address, name)
                    if full_path.endswith(filename):
                        final_path = copy.deepcopy(full_path)
        if final_path is not None:
            print(final_path)
            all_paths.append(final_path)
            if task == "ats":
                metric_names=['AlignScore']
            elif (ds_name in ["MMLU", "GSM8k"]):
                metric_names=["Accuracy"]
            elif task == "qa":
                metric_names=['AlignScore']
            elif task == "nmt":
                metric_names=['Comet']
            aggregated = True if ds_name in ["TruthfulQA", "TriviaQA"] else False
            df = get_table_short(final_path, dataset=ds_name, metric_names=metric_names, update_alignscore=False, aggregated=aggregated, return_score=False, upd=False, roc_auc=False, update_alignscore_sum=False, add_nll=False, mc_num=16)
            ind = np.argwhere(["P_i" in x or "MaximumSequenceProbability" in x for x in df[df.columns[-1:]].values.flatten()]).flatten()
            dfs[model][ds_name] = df.reset_index(drop=True)

../workdir/dllm_baselines/ats/GSAI-ML/LLaDA-1.5/['LM-Polygraph/samsum', 'simple_instruct']/2026-03-27/01-38-57/ue_manager_seed1
AlignScore:  0.8985490752978226
../workdir/dllm_baselines/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2026-03-27/10-12-46/ue_manager_seed1
Comet:  0.8578273457363248
../workdir/dllm_baselines/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt19', 'deen_simple_instruct']/2026-03-27/12-39-25/ue_manager_seed1
Comet:  0.8497599904537201
../workdir/dllm_baselines/qa/GSAI-ML/LLaDA-1.5/['LM-polygraph/coqa', 'simple_instruct']/2026-03-31/10-25-58/ue_manager_seed1
AlignScore:  0.736534931716822
../workdir/dllm_baselines/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/triviaqa', 'simple_instruct']/2026-03-26/23-37-58/ue_manager_seed1
AlignScore:  0.45934441404898096
../workdir/dllm_baselines/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/mmlu', 'simple_instruct']/2026-04-07/23-36-27/ue_manager_seed1
Accuracy:  0.656


In [179]:
def get_time(path, base=False):
    man = torch.load(path, weights_only=False)
    # print(man["time_stats"].keys())
    df = {}
    methods_diff = {
        "MSP": ["greedy_tokens"],
        "NFEUncertainty": ["NFEUncertainty", "greedy_tokens"],
        "MCMaskedNLL": ["MCMaskedNLL", "mc_masked_ll", "greedy_tokens"],
        "AvgDissimilarityFullTrace": ["AvgDissimilarityFullTrace", "greedy_sentence_similarity_full_trace", "greedy_tokens"],
        "CocoaMCMaskedNormalizedNLLCrossEncoderFullTraceNFE": ["CocoaMCMaskedNormalizedNLLCrossEncoderFullTraceNFE", "greedy_tokens", "greedy_sentence_similarity_full_trace", "mc_masked_ll"],
    }
    methods_base = {
        "MSP": ["greedy_tokens"],
        "MCMaskedNLL": ["MCMaskedNLL", "mc_masked_ll", "greedy_tokens"],
        "KernelLanguageEntropy": ["KernelLanguageEntropy", "semantic_matrix_entail", "sample_texts", "greedy_tokens"],
        "DegMat_NLI_score_entail": ["DegMat_NLI_score_entail", "semantic_matrix_entail", "sample_texts", "greedy_tokens"],
        "LUQ": ["LUQ", "semantic_matrix_entail_logits", "sample_texts", "greedy_tokens"],
        "CocoaMTE": ["CocoaMTE", "greedy_sentence_similarity", "sample_texts", "greedy_tokens"],
        "EigenScore": ["EigenScore", "sample_embeddings", "greedy_tokens"],
        "CocoaMCMaskedNormalizedNLL": ["CocoaMCMaskedNormalizedNLL", "sample_texts", "greedy_sentence_similarity", "mc_masked_ll", "greedy_tokens"],
        "TAD": [ "TAD", "greedy_tokens","attention_all"],
    }
    if base:
        methods = methods_base
    else:
        methods = methods_diff
    times = {}
    for method in methods.keys():
        times[method] = 0
        for stat in methods[method]:
            scores = np.array(man["time_stats"][stat])
            if len(scores) == 0:
                print(stat, scores)
            times[method] += np.array(scores[1:])
    return times

In [209]:
datasets = ['ats/xsum', 'ats/samsum', "ats/['cnn_dailymail', '3.0.0']", "nmt/['wmt14', 'fr-en']", "nmt/['wmt19', 'de-en']", "qa/keivalya/MedQuad-MedicalQnADataset", "qa/['truthful_qa', 'generation']", 'qa/coqa', 'qa/sciq', "qa/['trivia_qa', 'rc.nocontext']", "qa/['cais/mmlu', 'all']", "qa/['gsm8k', 'main']"]
datasets_names = ['XSUM', 'SamSum', 'CNN', "WMT14", "WMT19", "MedQUAD", "TruthfulQA", 'CoQA', 'SciQ', "TriviaQA", "MMLU", "GSM8k"]

models = ['GSAI-ML/LLaDA-1.5/']

paths = {
    "GSAI-ML/LLaDA-1.5/": {
        "SamSum": "../workdir/output_dllm_dec25_maskedll_final/ats/GSAI-ML/LLaDA-1.5/['LM-Polygraph/samsum', 'simple_instruct']/2026-03-25/01-04-55/ue_manager_seed1",
        "XSUM": "../workdir/output_dllm_dec25_maskedll_final/ats/GSAI-ML/LLaDA-1.5/['LM-Polygraph/xsum', 'simple_instruct']/2026-03-25/16-52-38/ue_manager_seed1",
        "WMT14": "../workdir/output_dllm_dec25_maskedll_final/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2026-03-25/04-09-53/ue_manager_seed1",
        "WMT19": "../workdir/output_dllm_dec25_maskedll_final/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt19', 'deen_simple_instruct']/2026-03-25/08-14-48/ue_manager_seed1",
        # "CoQA": "../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-polygraph/coqa', 'simple_instruct']/2026-03-25/00-53-35/ue_manager_seed1",
        "GSM8k": "../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/gsm8k', 'simple_instruct']/2026-03-24/14-28-05/ue_manager_seed1",
        "MMLU": "../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/mmlu', 'simple_instruct']/2026-03-25/15-16-53/ue_manager_seed1",
        # "TriviaQA": "../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/triviaqa', 'simple_instruct']/2026-03-25/05-21-47/ue_manager_seed1",
        # "TruthfulQA": "../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/truthfulqa', 'simple_instruct']/2026-03-25/07-52-54/ue_manager_seed1",
        
    }
}

dfs_time = {}
all_paths = []
for model in models:
    dfs_time[model] = []
    for ds, ds_name in zip(datasets, datasets_names):
        try:
            final_path = paths[model][ds_name]
        except:
            continue
        if final_path is not None:
            print(final_path)
            df = get_time(final_path, base=False)
            dfs_time[model].append(df)


../workdir/output_dllm_dec25_maskedll_final/ats/GSAI-ML/LLaDA-1.5/['LM-Polygraph/xsum', 'simple_instruct']/2026-03-25/16-52-38/ue_manager_seed1
../workdir/output_dllm_dec25_maskedll_final/ats/GSAI-ML/LLaDA-1.5/['LM-Polygraph/samsum', 'simple_instruct']/2026-03-25/01-04-55/ue_manager_seed1
../workdir/output_dllm_dec25_maskedll_final/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2026-03-25/04-09-53/ue_manager_seed1
../workdir/output_dllm_dec25_maskedll_final/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt19', 'deen_simple_instruct']/2026-03-25/08-14-48/ue_manager_seed1
../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/mmlu', 'simple_instruct']/2026-03-25/15-16-53/ue_manager_seed1
../workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/gsm8k', 'simple_instruct']/2026-03-24/14-28-05/ue_manager_seed1


In [202]:
methods = {k: [] for k in dfs_time[model][0].keys()}
for ds in dfs_time[model]:
    for m in ds.keys():
        methods[m].append(np.mean(ds[m]))

In [208]:
datasets = ['ats/xsum', 'ats/samsum', "ats/['cnn_dailymail', '3.0.0']", "nmt/['wmt14', 'fr-en']", "nmt/['wmt19', 'de-en']", "qa/keivalya/MedQuad-MedicalQnADataset", "qa/['truthful_qa', 'generation']", 'qa/coqa', 'qa/sciq', "qa/['trivia_qa', 'rc.nocontext']", "qa/['cais/mmlu', 'all']", "qa/['gsm8k', 'main']"]
datasets_names = ['XSUM', 'SamSum', 'CNN', "WMT14", "WMT19", "MedQUAD", "TruthfulQA", 'CoQA', 'SciQ', "TriviaQA", "MMLU", "GSM8k"]

models = ['GSAI-ML/LLaDA-1.5/']

paths = {
    "GSAI-ML/LLaDA-1.5/": {
        "SamSum": "../workdir/dllm_baselines/ats/GSAI-ML/LLaDA-1.5/['LM-Polygraph/samsum', 'simple_instruct']/2026-03-27/01-38-57/ue_manager_seed1",
        "WMT14": "../workdir/dllm_baselines/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2026-03-27/10-12-46/ue_manager_seed1",
        "WMT19": "../workdir/dllm_baselines/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt19', 'deen_simple_instruct']/2026-03-27/12-39-25/ue_manager_seed1",
        "TriviaQA": "../workdir/dllm_baselines/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/triviaqa', 'simple_instruct']/2026-03-26/23-37-58/ue_manager_seed1",
        "CoQA": "../workdir/dllm_baselines/qa/GSAI-ML/LLaDA-1.5/['LM-polygraph/coqa', 'simple_instruct']/2026-03-31/10-25-58/ue_manager_seed1",
        # "TruthfulQA": "../workdir/dllm_baselines/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/truthfulqa', 'simple_instruct']/2026-04-07/06-40-05/ue_manager_seed1",
        "MMLU": "../workdir/dllm_baselines/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/mmlu', 'simple_instruct']/2026-04-07/23-36-27/ue_manager_seed1",
    }
}

dfs_time = {}
for model in models:
    dfs_time[model] = []
    for ds, ds_name in zip(datasets, datasets_names):
        try:
            final_path = paths[model][ds_name]
        except:
            continue
        if final_path is not None:
            print(final_path)
            df = get_time(final_path, base=True)
            dfs_time[model].append(df)


../workdir/dllm_baselines/ats/GSAI-ML/LLaDA-1.5/['LM-Polygraph/samsum', 'simple_instruct']/2026-03-27/01-38-57/ue_manager_seed1
../workdir/dllm_baselines/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2026-03-27/10-12-46/ue_manager_seed1
../workdir/dllm_baselines/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt19', 'deen_simple_instruct']/2026-03-27/12-39-25/ue_manager_seed1
../workdir/dllm_baselines/qa/GSAI-ML/LLaDA-1.5/['LM-polygraph/coqa', 'simple_instruct']/2026-03-31/10-25-58/ue_manager_seed1
../workdir/dllm_baselines/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/triviaqa', 'simple_instruct']/2026-03-26/23-37-58/ue_manager_seed1
../workdir/dllm_baselines/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/mmlu', 'simple_instruct']/2026-04-07/23-36-27/ue_manager_seed1


In [204]:
methods_1 = {k: [] for k in dfs_time[model][0].keys()}
for ds in dfs_time[model]:
    for m in ds.keys():
        methods_1[m].append(np.mean(ds[m]))
methods.update(methods_1)

In [206]:
pd.DataFrame(methods)

,MSP,NFEUncertainty,MCMaskedNLL,AvgDissimilarityFullTrace,CocoaMCMaskedNormalizedNLLCrossEncoderFullTraceNFE,KernelLanguageEntropy,DegMat_NLI_score_entail,LUQ,CocoaMTE,EigenScore,CocoaMCMaskedNormalizedNLL,TAD
0,3.409134,3.350134,3.834915,4.029138,4.511135,36.861874,36.843956,36.843949,36.115871,36.034564,36.541605,4.034960
1,1.007402,0.998153,1.183573,1.068550,1.260518,10.919287,10.903128,10.903120,10.723759,10.740984,10.899885,1.242585
2,1.045063,1.060097,1.213586,1.132837,1.317426,11.485590,11.469584,11.469575,11.255279,11.277925,11.423760,1.254538
3,1.115448,0.619787,1.217382,0.653900,0.739860,10.675250,10.658275,10.658270,10.339035,10.348352,10.440893,1.121722


In [199]:
dfs_short = []
methods = None
model = models[0]

for key in dfs[model].keys():
    df_ = dfs[model][key]   
    if methods is None:
        methods = df_[df_.columns[-1]]
        # methods = [m for m in methods if "FullTrace" in m]
        methods = [m for m in methods if "InverseProgress" not in m]
    df_ = df_.set_index(df_.columns[-1], drop=False).reindex(methods)
    
    if len(dfs_short):
        dfs_short.append(df_[df_.columns[[0]]])
    else:
        dfs_short.append(df_[df_.columns[[1, 0]]])
dfs_short = pd.concat(dfs_short, axis=1).reset_index(drop=True)

In [92]:
df_all = pd.concat([dfs_short, dfs_short_new], axis=0).reset_index(drop=True)
df_all.loc[:, ('', "Mean")] = df_all[df_all.columns[1:]].mean(axis=1)
df_all.loc[:, ('', "Mean Rank")] = df_all[df_all.columns[1:-1]].apply(lambda x: rankdata(-x)).mean(axis=1).round(2)

In [93]:
df_all.sort_values(df_all.columns[-2], ascending=False).style.background_gradient(axis=0)

In [20]:
dfs_short.iloc[[4,67,77,30]].sort_values(dfs_short.columns[-2], ascending=False).style.background_gradient(axis=0)

In [24]:
dfs_short.iloc[[100, 132, 48, 189]].sort_values(dfs_short.columns[-2], ascending=False).style.background_gradient(axis=0)

## Dream

In [45]:
df1_dream = pd.read_csv("../workdir/dream_7b_cache_results_1.csv", header=[0, 1])
df1_dream = df1_dream[df1_dream.columns[:-2]]
df2_dream = pd.read_csv("../workdir/dream_7b_cache_results_2.csv", header=[0, 1])
df2_dream = df2_dream[df2_dream.columns[:-2]]
# df_dream = pd.concat([df1_dream, df2_dream], axis=0).reset_index(drop=True)
df_dream = pd.merge(df1_dream, df2_dream, left_on=[df1_dream.columns[0]], right_on=[df2_dream.columns[0]], how='outer')
df_dream = df_dream.drop(columns=[df2_dream.columns[0]])

df_dream.loc[:, ('', "Mean")] = df_dream[df_dream.columns[1:]].mean(axis=1)
df_dream.loc[:, ('', "Mean Rank")] = df_dream[df_dream.columns[1:-1]].apply(lambda x: rankdata(-x)).mean(axis=1).round(2)


In [46]:
df_dream[df_dream[df_dream.columns[0]] == "CocoaMCMaskedNormalizedNLLCrossEncoderFullTraceNFE"]


TriviaQA                  CoQA  \
                                                Method AlignScore AlignScore   
156  CocoaMCMaskedNormalizedNLLCrossEncoderFullTrac...   0.505855   0.417437   

        GSM8k     SamSum     WMT14     WMT19 TruthfulQA     MMLU            \
     Accuracy AlignScore     Comet     Comet AlignScore Accuracy      Mean   
156  0.288138   0.147419  0.268948 -0.013715   0.354894  0.01735  0.248291   

               
    Mean Rank  
156    141.62

In [47]:
df_dream.sort_values(df_dream.columns[-2], ascending=False).style.background_gradient(axis=0)

# MoE

In [12]:
import os
import copy
from scipy.stats import rankdata

base_path = "./workdir/output_dllm_dec25_maskedll_final_moe"

datasets = ['ats/xsum', 'ats/samsum', "ats/['cnn_dailymail', '3.0.0']", "nmt/['wmt14', 'fr-en']", "nmt/['wmt19', 'de-en']", "qa/keivalya/MedQuad-MedicalQnADataset", "qa/['truthful_qa', 'generation']", 'qa/coqa', 'qa/sciq', "qa/['trivia_qa', 'rc.nocontext']", "qa/['cais/mmlu', 'all']", "qa/['gsm8k', 'main']"]
datasets_names = ['XSUM', 'SamSum', 'CNN', "WMT14", "WMT19", "MedQUAD", "TruthfulQA", 'CoQA', 'SciQ', "TriviaQA", "MMLU", "GSM8k"]


models = ['inclusionAI/LLaDA-MoE-7B-A1B-Instruct']

paths = {
    "inclusionAI/LLaDA-MoE-7B-A1B-Instruct": {
        # # # "WMT14": "./workdir/output_dllm_dec25_maskedll/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2025-12-25/00-29-11/ue_manager_seed1",
        # # # "TriviaQA": "./workdir/output_dllm_dec25_maskedll/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/triviaqa', 'simple_instruct']/2025-12-24/20-02-47/ue_manager_seed1",
        # "SamSum": "../workdir/output_dllm_dec25_maskedll_final/ats/GSAI-ML/LLaDA-1.5/['LM-Polygraph/samsum', 'simple_instruct']/2026-01-13/22-40-23/ue_manager_seed1",
        # # "SamSum": "./workdir/output_dllm_dec25_maskedll_final/ats/GSAI-ML/LLaDA-1.5/['LM-Polygraph/samsum', 'simple_instruct']/2026-01-09/11-52-05/ue_manager_seed1",
        # "MMLU": "../workdir/output_dllm_dec25_maskedll_final/qa/{'path': 'GSAI-ML/LLaDA-1.5', 'ensemble': False, 'mc': False, 'mc_seeds': None, 'dropout_rate': None, 'type': 'DiffusionLLM', 'path_to_load_script': '/home/jovyan/artemvazhentsev/diffusion/llada/load_llada.py', 'load_model_args': {'device_map': 'auto'}, 'load_tokenizer_args': {}, 'device': 'cuda'}/['LM-Polygraph/mmlu', 'simple_instruct']/2026-01-14/10-33-48/ue_manager_seed1",
        # # "WMT14": "./workdir/output_dllm_dec25_maskedll_final/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2026-01-08/23-54-26/ue_manager_seed1",
        "WMT14": "../workdir/output_dllm_dec25_maskedll_final_moe/nmt/inclusionAI/LLaDA-MoE-7B-A1B-Instruct/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2026-02-18/10-30-10/ue_manager_seed1",
        # # "WMT19": "./workdir/output_dllm_dec25_maskedll_final/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt19', 'deen_simple_instruct']/2026-01-08/19-41-08/ue_manager_seed1",
        # "WMT19": "./workdir/output_dllm_dec25_maskedll_final/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt19', 'deen_simple_instruct']/2026-01-09/16-12-44/ue_manager_seed1",
        "WMT19": "../workdir/output_dllm_dec25_maskedll_final_moe/nmt/inclusionAI/LLaDA-MoE-7B-A1B-Instruct/['LM-Polygraph/wmt19', 'deen_simple_instruct']/2026-02-08/06-11-24/ue_manager_seed1",
        "TriviaQA": "../workdir/output_dllm_dec25_maskedll_final_moe/qa/inclusionAI/LLaDA-MoE-7B-A1B-Instruct/['LM-Polygraph/triviaqa', 'simple_instruct']/2026-02-10/15-00-48/ue_manager_seed1",
        # # # "SamSum": "./workdir/output_dllm_dec25_maskedll_final/ats/GSAI-ML/LLaDA-1.5/['LM-Polygraph/samsum', 'simple_instruct']/2026-01-02/04-58-43/ue_manager_seed1",
        "GSM8k": "../workdir/output_dllm_dec25_maskedll_final_moe/qa/inclusionAI/LLaDA-MoE-7B-A1B-Instruct/['LM-Polygraph/gsm8k', 'simple_instruct']/2026-02-18/10-26-00/ue_manager_seed1",
        # # "CoQA": "./workdir/output_dllm_dec25_maskedll_final/qa/GSAI-ML/LLaDA-1.5/['LM-polygraph/coqa', 'simple_instruct']/2026-01-11/02-41-07/ue_manager_seed1",
        # "CoQA": "../workdir/output_dllm_dec25_maskedll_final_moe/qa/inclusionAI/LLaDA-MoE-7B-A1B-Instruct/['LM-polygraph/coqa', 'simple_instruct']/2026-02-16/11-29-53/ue_manager_seed1",
        # "TruthfulQA": "../workdir/output_dllm_dec25_maskedll_final_moe/qa/{'path': 'inclusionAI/LLaDA-MoE-7B-A1B-Instruct', 'ensemble': False, 'mc': False, 'mc_seeds': None, 'dropout_rate': None, 'type': 'DiffusionLLM', 'path_to_load_script': '/home/artem.shelmanov/artemvazhentsev/uncertainty_dllm/llada/load_llada_moe.py', 'load_model_args': {'device_map': 'auto'}, 'load_tokenizer_args': {}, 'device': 'cuda', 'use_cache': False, 'mask_id': 156895}/['LM-Polygraph/truthfulqa', 'simple_instruct']/2026-02-07/10-26-07/ue_manager_seed1",
    }
}

filename = "ue_manager_seed1"
dfs = {}

all_paths = []
for model in models:
    dfs[model] = {}
    for ds, ds_name in zip(datasets, datasets_names):
        task = ds.split("/")[0]
        ds = '/'.join(ds.split("/")[1:])
        path = f"{base_path}/{task}/{model}/{ds}/"
        final_path = None
        if model in paths.keys() and ds_name in paths[model].keys():
            final_path = paths[model][ds_name]
        else:
            continue
            for address, dirs, files in os.walk(path):
                for name in files:
                    full_path = os.path.join(address, name)
                    if full_path.endswith(filename):
                        final_path = copy.deepcopy(full_path)
        if final_path is not None:
            print(final_path)
            all_paths.append(final_path)
            if task == "ats":
                metric_names=['AlignScore']
            elif (ds_name in ["MMLU", "GSM8k"]):
                metric_names=["Accuracy"]
            elif task == "qa":
                metric_names=['AlignScore']
            elif task == "nmt":
                metric_names=['Comet']
            aggregated = True if ds_name in ["TruthfulQA", "TriviaQA"] else False
            df = get_table_short(final_path, dataset=ds_name, metric_names=metric_names, update_alignscore=False, aggregated=aggregated, return_score=False, upd=False, roc_auc=False, update_alignscore_sum=False, add_nll=False, mc_num=16)
            ind = np.argwhere(["P_i" in x or "MaximumSequenceProbability" in x for x in df[df.columns[-1:]].values.flatten()]).flatten()
            dfs[model][ds_name] = df.reset_index(drop=True)

../workdir/output_dllm_dec25_maskedll_final_moe/nmt/inclusionAI/LLaDA-MoE-7B-A1B-Instruct/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2026-02-18/10-30-10/ue_manager_seed1
Comet:  0.857942321524024
../workdir/output_dllm_dec25_maskedll_final_moe/nmt/inclusionAI/LLaDA-MoE-7B-A1B-Instruct/['LM-Polygraph/wmt19', 'deen_simple_instruct']/2026-02-08/06-11-24/ue_manager_seed1
Comet:  0.8532064864188432
../workdir/output_dllm_dec25_maskedll_final_moe/qa/inclusionAI/LLaDA-MoE-7B-A1B-Instruct/['LM-Polygraph/triviaqa', 'simple_instruct']/2026-02-10/15-00-48/ue_manager_seed1
AlignScore:  0.443462045154025
../workdir/output_dllm_dec25_maskedll_final_moe/qa/inclusionAI/LLaDA-MoE-7B-A1B-Instruct/['LM-Polygraph/gsm8k', 'simple_instruct']/2026-02-18/10-26-00/ue_manager_seed1
Accuracy:  0.79


In [13]:
dfs_short = []
methods = None
model = models[0]

for key in dfs[model].keys():
    df_ = dfs[model][key]   
    if methods is None:
        methods = df_[df_.columns[-1]]
    df_ = df_.set_index(df_.columns[-1], drop=False).reindex(methods)
    
    if len(dfs_short):
        dfs_short.append(df_[df_.columns[[0]]])
    else:
        dfs_short.append(df_[df_.columns[[1, 0]]])
dfs_short = pd.concat(dfs_short, axis=1).reset_index(drop=True)
dfs_short.loc[:, ('', "Mean")] = dfs_short[dfs_short.columns[1:]].mean(axis=1)
dfs_short.loc[:, ('', "Mean Rank")] = dfs_short[dfs_short.columns[1:-1]].apply(lambda x: rankdata(-x)).mean(axis=1).round(2)

In [14]:
dfs_short.sort_values(dfs_short.columns[-2], ascending=False).style.background_gradient(axis=0)

In [10]:
path = "../workdir/output_dllm_dec25_maskedll_final_moe/qa/inclusionAI/LLaDA-MoE-7B-A1B-Instruct/['LM-polygraph/coqa', 'simple_instruct']/2026-02-16/11-29-53/ue_manager_seed1"

In [11]:
data = torch.load(path, weights_only=False)

### Test

In [7]:
import os

import copy
from scipy.stats import rankdata

base_path = "./workdir/output"

datasets = ["nmt/['wmt14', 'fr-en']", "qa/['gsm8k', 'main']"]
datasets_names = ["WMT14", "GSM8k"]

models = ['GSAI-ML/LLaDA-1.5/']

paths = {
    "GSAI-ML/LLaDA-1.5/": {
        # "WMT14": "/home/artem.shelmanov/artemvazhentsev/uncertainty_dllm_dream/uncertainty_dllm/workdir/output_test/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2026-05-08/14-14-06/ue_manager_seed1",
        # "WMT14": "/home/artem.shelmanov/artemvazhentsev/uncertainty_dllm_dream/uncertainty_dllm/workdir/output_test/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2026-07-21/13-10-18/ue_manager_seed1",
        # "WMT14": "/home/artem.shelmanov/artemvazhentsev/uncertainty_dllm_dream/uncertainty_dllm/workdir/output_test/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2026-07-22/10-22-29/ue_manager_seed1",
        "WMT14": "/home/artem.shelmanov/artemvazhentsev/uncertainty_dllm_dream/uncertainty_dllm/workdir/output_test/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2026-07-22/18-31-27/ue_manager_seed1",
        # "GSM8k": "/home/artem.shelmanov/artemvazhentsev/uncertainty_dllm_dream/uncertainty_dllm/workdir/output_test/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/gsm8k', 'simple_instruct']/2026-05-08/17-40-21/ue_manager_seed1",
    }
}

filename = "ue_manager_seed1"
dfs = {}

all_paths = []
for model in models:
    dfs[model] = {}
    for ds, ds_name in zip(datasets, datasets_names):
        task = ds.split("/")[0]
        ds = '/'.join(ds.split("/")[1:])
        path = f"{base_path}/{task}/{model}/{ds}/"
        final_path = None
        if model in paths.keys() and ds_name in paths[model].keys():
            final_path = paths[model][ds_name]
        else:
            continue
            for address, dirs, files in os.walk(path):
                for name in files:
                    full_path = os.path.join(address, name)
                    if full_path.endswith(filename):
                        final_path = copy.deepcopy(full_path)
        if final_path is not None:
            print(final_path)
            all_paths.append(final_path)
            if task == "ats":
                metric_names=['AlignScore']
            elif (ds_name in ["MMLU", "GSM8k"]):
                metric_names=["Accuracy"]
            elif task == "qa":
                metric_names=['AlignScore']
            elif task == "nmt":
                metric_names=['Comet']
            aggregated = True if ds_name in ["TruthfulQA", "TriviaQA"] else False
            df = get_table_short(final_path, dataset=ds_name, metric_names=metric_names, update_alignscore=False, aggregated=aggregated, return_score=False, upd=False, roc_auc=False, update_alignscore_sum=False, add_nll=False, mc_num=16)
            ind = np.argwhere(["P_i" in x or "MaximumSequenceProbability" in x for x in df[df.columns[-1:]].values.flatten()]).flatten()
            dfs[model][ds_name] = df.reset_index(drop=True)

/home/artem.shelmanov/artemvazhentsev/uncertainty_dllm_dream/uncertainty_dllm/workdir/output_test/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2026-07-22/18-31-27/ue_manager_seed1
Comet:  0.8698779284954071


In [8]:
dfs_short = []
methods = None
model = models[0]

for key in dfs[model].keys():
    df_ = dfs[model][key]   
    if methods is None:
        methods = df_[df_.columns[-1]]
    df_ = df_.set_index(df_.columns[-1], drop=False).loc[methods]
    
    if len(dfs_short):
        dfs_short.append(df_[df_.columns[[0]]])
    else:
        dfs_short.append(df_[df_.columns[[1, 0]]])
dfs_short1 = pd.concat(dfs_short, axis=1).reset_index(drop=True)
# dfs_short1.loc[:, ('', "Mean")] = dfs_short1[dfs_short1.columns[1:]].mean(axis=1)
# dfs_short1.loc[:, ('', "Mean Rank")] = dfs_short1[dfs_short1.columns[1:-1]].apply(lambda x: rankdata(-x)).mean(axis=1).round(2)

In [9]:
dfs_short1.sort_values(dfs_short1.columns[-1]).style.background_gradient(axis=0)

In [10]:
import os

import copy
from scipy.stats import rankdata

base_path = "./workdir/output"

datasets = ["nmt/['wmt14', 'fr-en']", "qa/['gsm8k', 'main']"]
datasets_names = ["WMT14", "GSM8k"]

models = ['GSAI-ML/LLaDA-1.5/']

paths = {
    "GSAI-ML/LLaDA-1.5/": {
        # "WMT14": "/home/artem.shelmanov/artemvazhentsev/uncertainty_dllm/workdir/output_test/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2026-05-08/13-45-14/ue_manager_seed1",
        # "WMT14": "/home/artem.shelmanov/artemvazhentsev/uncertainty_dllm/workdir/output_test/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2026-07-22/10-34-01/ue_manager_seed1",
        "WMT14": "/home/artem.shelmanov/artemvazhentsev/uncertainty_dllm/workdir/output_test/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2026-07-22/19-23-30/ue_manager_seed1",
        # "GSM8k": "/home/artem.shelmanov/artemvazhentsev/uncertainty_dllm_dream/uncertainty_dllm/workdir/output_test/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/gsm8k', 'simple_instruct']/2026-05-08/17-40-21/ue_manager_seed1",
    }
}

filename = "ue_manager_seed1"
dfs = {}

all_paths = []
for model in models:
    dfs[model] = {}
    for ds, ds_name in zip(datasets, datasets_names):
        task = ds.split("/")[0]
        ds = '/'.join(ds.split("/")[1:])
        path = f"{base_path}/{task}/{model}/{ds}/"
        final_path = None
        if model in paths.keys() and ds_name in paths[model].keys():
            final_path = paths[model][ds_name]
        else:
            continue
            for address, dirs, files in os.walk(path):
                for name in files:
                    full_path = os.path.join(address, name)
                    if full_path.endswith(filename):
                        final_path = copy.deepcopy(full_path)
        if final_path is not None:
            print(final_path)
            all_paths.append(final_path)
            if task == "ats":
                metric_names=['AlignScore']
            elif (ds_name in ["MMLU", "GSM8k"]):
                metric_names=["Accuracy"]
            elif task == "qa":
                metric_names=['AlignScore']
            elif task == "nmt":
                metric_names=['Comet']
            aggregated = True if ds_name in ["TruthfulQA", "TriviaQA"] else False
            df = get_table_short(final_path, dataset=ds_name, metric_names=metric_names, update_alignscore=False, aggregated=aggregated, return_score=False, upd=False, roc_auc=False, update_alignscore_sum=False, add_nll=False, mc_num=16)
            ind = np.argwhere(["P_i" in x or "MaximumSequenceProbability" in x for x in df[df.columns[-1:]].values.flatten()]).flatten()
            dfs[model][ds_name] = df.reset_index(drop=True)

/home/artem.shelmanov/artemvazhentsev/uncertainty_dllm/workdir/output_test/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2026-07-22/19-23-30/ue_manager_seed1
Comet:  0.8698779284954071


In [11]:
dfs_short = []
methods = None
model = models[0]

for key in dfs[model].keys():
    df_ = dfs[model][key]   
    if methods is None:
        methods = df_[df_.columns[-1]]
    df_ = df_.set_index(df_.columns[-1], drop=False).loc[methods]
    
    if len(dfs_short):
        dfs_short.append(df_[df_.columns[[0]]])
    else:
        dfs_short.append(df_[df_.columns[[1, 0]]])
dfs_short = pd.concat(dfs_short, axis=1).reset_index(drop=True)
# dfs_short.loc[:, ('', "Mean")] = dfs_short[dfs_short.columns[1:]].mean(axis=1)
# dfs_short.loc[:, ('', "Mean Rank")] = dfs_short[dfs_short.columns[1:-1]].apply(lambda x: rankdata(-x)).mean(axis=1).round(2)

In [12]:
dfs_short.sort_values(dfs_short.columns[-1]).style.background_gradient(axis=0)

In [13]:
df = pd.concat(
    [
        dfs_short,
        dfs_short1.add_suffix("_dream"),
    ],
    axis=1,
).reset_index(drop=True)

In [14]:
from scipy.stats import spearmanr

spearmanr(df[df.columns[1]], df[df.columns[3]]).statistic

1.0

In [15]:
np.argsort(df[df.columns[1]] - df[df.columns[3]])[-10:]

500    168
501    167
502    166
503    165
504    164
505    163
506    162
507    161
508    174
509    509
dtype: int64

In [16]:
df.iloc[[504]]

WMT14                 WMT14_dream            
               Method     Comet      Method_dream Comet_dream
504  TraceEntropy_avg  0.529976  TraceEntropy_avg    0.529976

In [17]:
dfs_short.sort_values(dfs_short.columns[-1], ascending=False).iloc[:10].style.background_gradient(axis=0)

In [18]:
dfs_short1.sort_values(dfs_short.columns[-1], ascending=False).iloc[:10].style.background_gradient(axis=0)

In [20]:
dfs_short.iloc[[228]].style.background_gradient(axis=0)

## Dream

In [ ]:
import os

import copy
from scipy.stats import rankdata

base_path = "./workdir/output"

datasets = ["nmt/['wmt14', 'fr-en']", "qa/['gsm8k', 'main']"]
datasets_names = ["WMT14", "GSM8k"]

models = ['Dream']

paths = {
    "Dream": {
        "WMT14": "/l/users/artem.shelmanov/artemvazhentsev/uncertainty_dllm_dream/uncertainty_dllm/workdir/output_test/nmt/Dream-org/Dream-v0-Instruct-7B/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2026-07-22/18-16-04/ue_manager_seed1",
    }
}

filename = "ue_manager_seed1"
dfs = {}

all_paths = []
for model in models:
    dfs[model] = {}
    for ds, ds_name in zip(datasets, datasets_names):
        task = ds.split("/")[0]
        ds = '/'.join(ds.split("/")[1:])
        path = f"{base_path}/{task}/{model}/{ds}/"
        final_path = None
        if model in paths.keys() and ds_name in paths[model].keys():
            final_path = paths[model][ds_name]
        else:
            continue
            for address, dirs, files in os.walk(path):
                for name in files:
                    full_path = os.path.join(address, name)
                    if full_path.endswith(filename):
                        final_path = copy.deepcopy(full_path)
        if final_path is not None:
            print(final_path)
            all_paths.append(final_path)
            if task == "ats":
                metric_names=['AlignScore']
            elif (ds_name in ["MMLU", "GSM8k"]):
                metric_names=["Accuracy"]
            elif task == "qa":
                metric_names=['AlignScore']
            elif task == "nmt":
                metric_names=['Comet']
            aggregated = True if ds_name in ["TruthfulQA", "TriviaQA"] else False
            df = get_table_short(final_path, dataset=ds_name, metric_names=metric_names, update_alignscore=False, aggregated=aggregated, return_score=False, upd=False, roc_auc=False, update_alignscore_sum=False, add_nll=False, mc_num=16)
            ind = np.argwhere(["P_i" in x or "MaximumSequenceProbability" in x for x in df[df.columns[-1:]].values.flatten()]).flatten()
            dfs[model][ds_name] = df.reset_index(drop=True)

True
/l/users/artem.shelmanov/artemvazhentsev/uncertainty_dllm_dream/uncertainty_dllm/workdir/output_test/nmt/Dream-org/Dream-v0-Instruct-7B/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2026-07-22/18-16-04/ue_manager_seed1
Comet:  0.8518353804945946
True


In [22]:
dfs_short = []
methods = None
model = models[0]

for key in dfs[model].keys():
    df_ = dfs[model][key]   
    if methods is None:
        methods = df_[df_.columns[-1]]
    df_ = df_.set_index(df_.columns[-1], drop=False).loc[methods]
    
    if len(dfs_short):
        dfs_short.append(df_[df_.columns[[0]]])
    else:
        dfs_short.append(df_[df_.columns[[1, 0]]])
dfs_short = pd.concat(dfs_short, axis=1).reset_index(drop=True)
# dfs_short.loc[:, ('', "Mean")] = dfs_short[dfs_short.columns[1:]].mean(axis=1)
# dfs_short.loc[:, ('', "Mean Rank")] = dfs_short[dfs_short.columns[1:-1]].apply(lambda x: rankdata(-x)).mean(axis=1).round(2)

In [23]:
dfs_short.sort_values(dfs_short.columns[-1]).style.background_gradient(axis=0)

## Refusion

In [24]:
import os

import copy
from scipy.stats import rankdata

base_path = "./workdir/output"

datasets = ["nmt/['wmt14', 'fr-en']", "qa/['gsm8k', 'main']"]
datasets_names = ["WMT14", "GSM8k"]

models = ['ReFusion']

paths = {
    "ReFusion": {
        "WMT14": "/home/artem.shelmanov/artemvazhentsev/uncertainty_dllm_dream/uncertainty_dllm/workdir/output_test/nmt/GSAI-ML/ReFusion/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2026-07-22/19-11-36/ue_manager_seed1",
    }
}

filename = "ue_manager_seed1"
dfs = {}

all_paths = []
for model in models:
    dfs[model] = {}
    for ds, ds_name in zip(datasets, datasets_names):
        task = ds.split("/")[0]
        ds = '/'.join(ds.split("/")[1:])
        path = f"{base_path}/{task}/{model}/{ds}/"
        final_path = None
        if model in paths.keys() and ds_name in paths[model].keys():
            final_path = paths[model][ds_name]
        else:
            continue
            for address, dirs, files in os.walk(path):
                for name in files:
                    full_path = os.path.join(address, name)
                    if full_path.endswith(filename):
                        final_path = copy.deepcopy(full_path)
        if final_path is not None:
            print(final_path)
            all_paths.append(final_path)
            if task == "ats":
                metric_names=['AlignScore']
            elif (ds_name in ["MMLU", "GSM8k"]):
                metric_names=["Accuracy"]
            elif task == "qa":
                metric_names=['AlignScore']
            elif task == "nmt":
                metric_names=['Comet']
            aggregated = True if ds_name in ["TruthfulQA", "TriviaQA"] else False
            df = get_table_short(final_path, dataset=ds_name, metric_names=metric_names, update_alignscore=False, aggregated=aggregated, return_score=False, upd=False, roc_auc=False, update_alignscore_sum=False, add_nll=False, mc_num=16)
            ind = np.argwhere(["P_i" in x or "MaximumSequenceProbability" in x for x in df[df.columns[-1:]].values.flatten()]).flatten()
            dfs[model][ds_name] = df.reset_index(drop=True)

/home/artem.shelmanov/artemvazhentsev/uncertainty_dllm_dream/uncertainty_dllm/workdir/output_test/nmt/GSAI-ML/ReFusion/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2026-07-22/19-11-36/ue_manager_seed1
Comet:  0.8551376205682755


In [25]:
dfs_short = []
methods = None
model = models[0]

for key in dfs[model].keys():
    df_ = dfs[model][key]   
    if methods is None:
        methods = df_[df_.columns[-1]]
    df_ = df_.set_index(df_.columns[-1], drop=False).loc[methods]
    
    if len(dfs_short):
        dfs_short.append(df_[df_.columns[[0]]])
    else:
        dfs_short.append(df_[df_.columns[[1, 0]]])
dfs_short = pd.concat(dfs_short, axis=1).reset_index(drop=True)
# dfs_short.loc[:, ('', "Mean")] = dfs_short[dfs_short.columns[1:]].mean(axis=1)
# dfs_short.loc[:, ('', "Mean Rank")] = dfs_short[dfs_short.columns[1:-1]].apply(lambda x: rankdata(-x)).mean(axis=1).round(2)

In [26]:
dfs_short.sort_values(dfs_short.columns[-1]).style.background_gradient(axis=0)

In [27]:
import os

import copy
from scipy.stats import rankdata

base_path = "./workdir/output"

datasets = ["nmt/['wmt14', 'fr-en']", "qa/['gsm8k', 'main']"]
datasets_names = ["WMT14", "GSM8k"]

models = ['ReFusion']

paths = {
    "ReFusion": {
        "WMT14": "/home/artem.shelmanov/artemvazhentsev/uncertainty_dllm/workdir/output_test/nmt/GSAI-ML/ReFusion/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2026-07-22/19-36-38/ue_manager_seed1",
    }
}

filename = "ue_manager_seed1"
dfs = {}

all_paths = []
for model in models:
    dfs[model] = {}
    for ds, ds_name in zip(datasets, datasets_names):
        task = ds.split("/")[0]
        ds = '/'.join(ds.split("/")[1:])
        path = f"{base_path}/{task}/{model}/{ds}/"
        final_path = None
        if model in paths.keys() and ds_name in paths[model].keys():
            final_path = paths[model][ds_name]
        else:
            continue
            for address, dirs, files in os.walk(path):
                for name in files:
                    full_path = os.path.join(address, name)
                    if full_path.endswith(filename):
                        final_path = copy.deepcopy(full_path)
        if final_path is not None:
            print(final_path)
            all_paths.append(final_path)
            if task == "ats":
                metric_names=['AlignScore']
            elif (ds_name in ["MMLU", "GSM8k"]):
                metric_names=["Accuracy"]
            elif task == "qa":
                metric_names=['AlignScore']
            elif task == "nmt":
                metric_names=['Comet']
            aggregated = True if ds_name in ["TruthfulQA", "TriviaQA"] else False
            df = get_table_short(final_path, dataset=ds_name, metric_names=metric_names, update_alignscore=False, aggregated=aggregated, return_score=False, upd=False, roc_auc=False, update_alignscore_sum=False, add_nll=False, mc_num=16)
            ind = np.argwhere(["P_i" in x or "MaximumSequenceProbability" in x for x in df[df.columns[-1:]].values.flatten()]).flatten()
            dfs[model][ds_name] = df.reset_index(drop=True)

/home/artem.shelmanov/artemvazhentsev/uncertainty_dllm/workdir/output_test/nmt/GSAI-ML/ReFusion/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2026-07-22/19-36-38/ue_manager_seed1
Comet:  0.8551376205682755


In [28]:
dfs_short = []
methods = None
model = models[0]

for key in dfs[model].keys():
    df_ = dfs[model][key]   
    if methods is None:
        methods = df_[df_.columns[-1]]
    df_ = df_.set_index(df_.columns[-1], drop=False).loc[methods]
    
    if len(dfs_short):
        dfs_short.append(df_[df_.columns[[0]]])
    else:
        dfs_short.append(df_[df_.columns[[1, 0]]])
dfs_short = pd.concat(dfs_short, axis=1).reset_index(drop=True)
# dfs_short.loc[:, ('', "Mean")] = dfs_short[dfs_short.columns[1:]].mean(axis=1)
# dfs_short.loc[:, ('', "Mean Rank")] = dfs_short[dfs_short.columns[1:-1]].apply(lambda x: rankdata(-x)).mean(axis=1).round(2)

In [29]:
dfs_short.sort_values(dfs_short.columns[-1]).style.background_gradient(axis=0)